# CS781 Phase 6 (abstractive legal summarisation) — PART 1: data, schema, validation split, compatibility checks, baselines, long-document study
**Sections of the specification covered here:** 1 environment, 2 dataset loading + schema verification, 3 exploration (document / summary lengths, missing values, duplicate ids), 4 reproducible train / validation split, 5 checkpoint discovery + compatibility with generation, 6 tokenisation, 7 baselines (extractive + zero-shot Phase 1 GPT-2: reference points only), 8 long-document study (input coverage of every strategy without a model + a neural study on GPT-2: head / tail / head+tail / importance selection / section-aware / shorter truncation, then hierarchical chunk -> summary inference), 16 output. It writes `best_long_doc_strategy.json` for the later parts.

**Attach:** the Phase 1 datasets `cs781-phase1-v3-gpt2-decoder`, `cs781-phase1-v3-encoder-modernbert`, `cs781-phase1-v3-t5-encdec` (+ `cs781-phase1-v3-bilstm-recurrent` if it exists; everything works without it). Secrets (Add-ons > Secrets, 'Attached' ON): `HF_TOKEN` (needed), `KAGGLE_USERNAME` + `KAGGLE_KEY` (optional, legacy key; only for the optional dataset backup). Settings: **Internet ON** (data, `bert-score` and its scoring model), **GPU T4 x1**.
**Estimated time (estimate):** about 3-3.5 h on a T4 (the neural study: 6 x ~20 min training + generation; the rest < 30 min). CPU-only cells: config, credentials, data, split, tokenisation, baselines, input coverage. Optional: `LD_HIER` (hierarchical inference), the strategy list `LD_STRATEGIES`.
**Output** (`/kaggle/working/p6`, restored by the next parts): `split.json`, `compat_report.csv`, `baselines.csv`, `longdoc_input_coverage.csv`, `longdoc_study.csv`, `best_long_doc_strategy.json`, the study runs (ablations, never counted) and the ledger.

## What was and was not tested (honest status)
The build sandbox had **no PyTorch, transformers, tokenizers or nbformat, and no access to download them**. Therefore:
- **Executed in the sandbox** (with stand-ins for `torch` / `tokenizers` / `datasets` / `kaggle_secrets` / the Kaggle CLI, synthetic data in the real file layout, `input` / `getpass` made to raise): the config, setup, credentials, evaluation toolkit (ROUGE, key facts, validator), data loading + schema detection + Hugging Face mock + local-file fallback, the validation split, tokenisation logic (with a stand-in tokenizer), the long-document input strategies, the baselines + input-coverage study, restore / merge, the comparison table, the rule check, the analysis writer, backup (fake CLI) and output cells.
- **NOT executed anywhere yet**: every cell that builds or trains a model or generates text (the model wrappers, the techniques, the trainer, generation, distillation). They were checked statically only (syntax, names defined before use). **Run Part 0 (the self-test, Kaggle CPU, no GPU quota) first**: it executes all of that code on tiny models of all four families.
- Never tested: the real dataset, the real Phase 1 checkpoints, GPU speed / memory, real metric values. Every time in this notebook is an **estimate**.


In [ ]:
# ================= CELL 0 — CONFIG  [RUNS NOW]. Edit this cell only; nothing below needs editing (part-specific switches are in the PART SETTINGS cell). =================
PART = "P1"                                                      # which notebook of the chain this is (written into every ledger row)
SELFTEST = False                                                     # True only in the Part 0 self-test notebook (tiny synthetic models and data)
REQUIRE_RESTORE = False                                              # True = this part needs an earlier part's notebook output attached (it stops early otherwise)
HF_DATASET = "Exploration-Lab/CS-781-Capstone"                       # gated Hugging Face dataset repo (accept its conditions on the page first)
HF_CONFIG = "phase_6"                                                # the Phase 6 subset / config
DATA_SOURCE = "hf"                                                   # "hf" = load_dataset(HF_DATASET, HF_CONFIG); "files" = read Train/train.json + Test/test.json under LOCAL_DATA_DIR
LOCAL_DATA_DIR = None                                                # e.g. "/kaggle/input/phase-6-summ"; also the FALLBACK if the Hugging Face load fails (None = search /kaggle/input for train.json / test.json)
ID_KEY, DOC_KEY, SUM_KEY = None, None, None                          # field names; None = detect them from the data (expected: id / document / summary)
PRED_KEY = "summary"                                                 # key of the submission objects: [{"id": ..., "summary": "..."}]
VAL_FRACTION, VAL_MIN, VAL_MAX, SPLIT_SEED = 0.10, 100, 1000, 781    # held-out validation split made from the public training data (deterministic: a hash of SPLIT_SEED + document text)
VAL_EVAL_CAP = 400                                                   # validation documents generated per experiment (the SAME fixed subset for every run, so runs are comparable)
CKPT_PATHS = {"gpt2_new": None, "modernbert": None, "t5": None, "bilstm": None}   # path to each Phase 1 final_model.pt; None = find it automatically under /kaggle/input
TOKENIZER_PATH = None                                                # path to legal_bpe_32k_mask.json; None = find it automatically
DISCOVER_MODELS = True                                               # scan /kaggle/input for Phase 1 checkpoints (the self-test turns this off)
P6_WORK = "/kaggle/working/p6"                                       # results, ledger, checkpoints, submissions (this folder IS the notebook output the next part restores)
SAVE_MODELS = True                                                   # save a checkpoint per experiment (adapter-only for parameter-efficient methods)
RERUN = False                                                        # False = a run whose result + checkpoint already exist (restored) is NOT trained again
BERTSCORE = True                                                     # compute BERTScore F1 (needs Internet ON to pip-install bert-score and download its scoring model the first time)
BERTSCORE_LANG, BERTSCORE_MODEL, BERTSCORE_RESCALE = "en", None, False   # library default for English (None = the library's default model); NOT necessarily the official evaluator's setting
ROUGE_STEMMER = True                                                 # Porter stemming in ROUGE (rouge_score library if installed, otherwise the built-in implementation)
TEST_GEN = True                                                      # every experiment also writes its own test-set submission (needed for 'six submissions')
TEST_GEN_MAX_MIN = 60                                                # cap on minutes for one run's test-set generation; if the estimate is larger, the run's test file is skipped (and says so)


In [ ]:
# ================= CELL 1 — ENVIRONMENT SETUP  [RUNS NOW]: imports, seeds, device, folders, optional metric libraries =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, shutil, subprocess, importlib, traceback, contextlib
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"     # fp16 autocast on the GPU (the T4 has no bf16); T5 runs in fp32 unless T5_FP16 = True (PART SETTINGS)
WORK = Path(P6_WORK)
for sub in ("runs", "cache", "submissions", "checkpoints"): (WORK / sub).mkdir(parents=True, exist_ok=True)
SUBDIR, CKPT, LEDGER = WORK / "submissions", WORK / "checkpoints", WORK / "ledger.csv"

def pip_install(pkg, module):
    """install a pure-software dependency (no model weights) if it is missing; returns True if importable afterwards"""
    try: importlib.import_module(module); return True
    except ImportError: pass
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True, timeout=600)
        importlib.invalidate_caches(); importlib.import_module(module); return True
    except Exception as e:
        print(f"could not install {pkg} ({type(e).__name__}); continuing without it"); return False
HAVE_ROUGE_LIB = pip_install("rouge-score", "rouge_score")
HAVE_BERTSCORE_LIB = pip_install("bert-score", "bert_score") if BERTSCORE else False
import transformers
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU unless this is the self-test)", "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| rouge_score lib:", HAVE_ROUGE_LIB, "| bert_score lib:", HAVE_BERTSCORE_LIB, "| work dir:", WORK)


In [ ]:
# ===== CREDENTIALS CHECK  [RUNS NOW; it never asks you to type anything]: reads HF_TOKEN (and optionally KAGGLE_USERNAME / KAGGLE_KEY) from Kaggle Secrets, verifies them, and stops early with the exact fix =====
# Needs Internet ON. Add the Secrets under Add-ons > Secrets with 'Attached' ON for THIS notebook (a committed run only sees attached secrets). The Hugging Face token is not needed when DATA_SOURCE = "files".
import os, subprocess, shutil
REQUIRE_HF, REQUIRE_KAGGLE = (DATA_SOURCE == "hf"), False        # the Kaggle key is only needed for the optional backup upload (set REQUIRE_KAGGLE = True to insist)
def get_secret(name):
    v = os.environ.get(name)
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        except Exception: v = None
    return (v or "").strip().strip("'\"") or None
problems, report = [], []
hf = get_secret("HF_TOKEN"); HF_OK = False
if DATA_SOURCE != "hf": report.append("Hugging Face token : not needed (DATA_SOURCE = 'files')"); HF_OK = True
elif not hf: report.append("Hugging Face token : MISSING"); problems.append("HF_TOKEN is missing: add a Secret named HF_TOKEN (a token of the account that was approved for the dataset), or set DATA_SOURCE = 'files'")
else:
    try:
        from huggingface_hub import HfApi; api = HfApi(); me = api.whoami(token=hf); api.auth_check(HF_DATASET, repo_type="dataset", token=hf); HF_OK = True; os.environ["HF_TOKEN"] = hf
        report.append(f"Hugging Face token : OK (user '{me.get('name')}', access to {HF_DATASET} confirmed, {len(hf)} characters)")
    except Exception as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        why = "Hugging Face does not recognise it (wrong, expired or revoked: create a new classic 'Read' token and update the Secret)" if code == 401 else "valid token, but this account has no approved access to the gated dataset" if code == 403 else f"{type(e).__name__}: {str(e)[:100]}"
        report.append(f"Hugging Face token : REFUSED (HTTP {code}): {why}"); problems.append(f"HF_TOKEN does not work: {why}")
ku, kk = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY"); KAGGLE_OK = False
if not (ku and kk): report.append("Kaggle credentials  : not provided (the backup upload will be a dry run)"); (problems.append("KAGGLE_USERNAME / KAGGLE_KEY are missing") if REQUIRE_KAGGLE else None)
elif not shutil.which("kaggle"): report.append("Kaggle credentials  : cannot be verified (the kaggle command is not installed)"); (problems.append("the kaggle command is not installed") if REQUIRE_KAGGLE else None)
else:
    try:
        r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=120, env={**os.environ, "KAGGLE_USERNAME": ku, "KAGGLE_KEY": kk})
        if r.returncode == 0: KAGGLE_OK = True; os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = ku, kk; report.append(f"Kaggle credentials  : OK (user '{ku}')")
        else: report.append(f"Kaggle credentials  : REFUSED ({(r.stdout + r.stderr).strip()[:100]})"); (problems.append("the Kaggle username/key were refused (use the LEGACY API key; Internet ON?)") if REQUIRE_KAGGLE else None)
    except Exception as e: report.append(f"Kaggle credentials  : not verified ({type(e).__name__})")
print("\n".join(report))
blocking = [p for p in problems if (p.startswith("HF_TOKEN") and REQUIRE_HF) or (not p.startswith("HF_TOKEN") and REQUIRE_KAGGLE)]
if blocking: raise RuntimeError("STOPPING EARLY (before any long step):\n  - " + "\n  - ".join(blocking) + "\nFix: Add-ons > Secrets > add the missing secret(s), switch 'Attached' ON for this notebook, and run again.")
print("\nCredentials ready.")


In [ ]:
# ================= CELL 3 — EVALUATION TOOLKIT  [RUNS NOW; CPU]. Every metric is defined here, in the open. The official evaluator's BERTScore model / settings are NOT known, so nothing here claims to reproduce the official number. =================
# PRIMARY  bertscore_f1 : the bert-score library, lang=BERTSCORE_LANG, model BERTSCORE_MODEL (None = library default for English), rescale_with_baseline=BERTSCORE_RESCALE. Labelled 'default setting, not necessarily the official one'.
#                         The scoring model is downloaded by the library: it is a METRIC, never part of a submission. Its encoder truncates long texts (about 510 sub-words). If the library or its model is unavailable: 'BERTScore: not computed' (no substitute).
# SECONDARY rouge1_f / rougeL_f : F1 of unigram overlap / of the longest common subsequence over the whole summary. rouge_score library (use_stemmer=ROUGE_STEMMER) if installed, else the built-in implementation below
#                         (lower-case, non-alphanumerics -> space, optional Porter stemming via nltk): ROUGE_IMPL says which one produced the numbers.
# QUALITATIVE (all heuristics, defined below): key-fact coverage, decision agreement, unsupported (possibly hallucinated) numbers / names, repetition, length ratio.
ROUGE_IMPL = "rouge_score" if HAVE_ROUGE_LIB else "builtin"
_PORTER = None
if ROUGE_IMPL == "builtin" and ROUGE_STEMMER:
    try:
        from nltk.stem.porter import PorterStemmer; _PORTER = PorterStemmer()
    except Exception: ROUGE_IMPL = "builtin-nostem"
def rouge_tokens(t):
    toks = re.sub(r"[^a-z0-9]+", " ", (t or "").lower()).split()
    return [_PORTER.stem(w) if (_PORTER is not None and len(w) > 3) else w for w in toks]
def lcs_len(a, b):
    """length of the longest common subsequence, vectorised row by row: row_new = cummax(max(row_prev, (row_prev shifted) + match))"""
    if not a or not b: return 0
    vocab = {w: i for i, w in enumerate(set(a) | set(b))}; bb = np.array([vocab[w] for w in b]); prev = np.zeros(len(b) + 1, dtype=np.int32)
    for w in a:
        m = (bb == vocab[w]); cand = np.maximum(prev[1:], prev[:-1] + m); prev = np.concatenate([[0], np.maximum.accumulate(cand)]).astype(np.int32)
    return int(prev[-1])
def _f(p, r): return 0.0 if p + r == 0 else 2 * p * r / (p + r)
def rouge_builtin(pred, ref):
    a, b = rouge_tokens(pred), rouge_tokens(ref)
    if not a or not b: return 0.0, 0.0
    ov = sum((Counter(a) & Counter(b)).values()); r1 = _f(ov / len(a), ov / len(b)); l = lcs_len(a, b); return r1, _f(l / len(a), l / len(b))
_ROUGE_SCORER = None
def rouge_pair(pred, ref):
    global _ROUGE_SCORER
    if ROUGE_IMPL == "rouge_score":
        if _ROUGE_SCORER is None:
            from rouge_score import rouge_scorer; _ROUGE_SCORER = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=ROUGE_STEMMER)
        s = _ROUGE_SCORER.score(ref, pred); return s["rouge1"].fmeasure, s["rougeL"].fmeasure
    return rouge_builtin(pred, ref)

_BS = {"scorer": None, "failed": None}
def bertscore_f1(preds, refs, batch_size=16):
    """per-document BERTScore F1 (numpy array), or None when it cannot be computed (the reason is printed once)"""
    if not BERTSCORE or not HAVE_BERTSCORE_LIB:
        if _BS["failed"] is None: _BS["failed"] = "disabled (BERTSCORE=False)" if not BERTSCORE else "bert-score library not installed (Internet off?)"; print("BERTScore: not computed:", _BS["failed"])
        return None
    if _BS["failed"]: return None
    try:
        if _BS["scorer"] is None:
            from bert_score import BERTScorer
            _BS["scorer"] = BERTScorer(lang=BERTSCORE_LANG, model_type=BERTSCORE_MODEL, rescale_with_baseline=BERTSCORE_RESCALE, device=str(device))
            print(f"BERTScore scorer: model {_BS['scorer'].model_type} | layer {_BS['scorer'].num_layers} | rescale {BERTSCORE_RESCALE} (library default setting, NOT necessarily the official one)")
        P, R, F_ = _BS["scorer"].score([p if p.strip() else "." for p in preds], list(refs), batch_size=batch_size); return F_.cpu().numpy().astype(float)
    except Exception as e:
        _BS["failed"] = f"{type(e).__name__}: {str(e)[:150]}"; print("BERTScore: not computed:", _BS["failed"]); return None
def bertscore_setting():
    s = _BS.get("scorer"); return f"{s.model_type}, layer {s.num_layers}, rescale={BERTSCORE_RESCALE}" if s is not None else "not computed"

# ---- key facts: the heuristic 'important legal facts' of a text (numbers / dates, statute references, acts, multi-word proper names, decision words) ----
DECISION_WORDS = ("allowed", "dismissed", "acquitted", "convicted", "quashed", "set aside", "upheld", "remanded", "affirmed", "reversed", "sentenced", "rejected", "granted", "disposed of", "partly allowed", "modified")
_STOPCAP = set("The This That These Those It In On At For By With From And But Or If When Where While Whereas Therefore Thus Hence However Further Also Accordingly Mr Mrs Ms Shri Smt Dr Learned Hon He She They We I A An As Of To Be No Not Our Their His Her Its Court Section Article Act".split())
def key_facts(text):
    t = text or ""; facts = set()
    for m in re.finditer(r"\b\d[\d,./-]*\d\b|\b\d\b", t): facts.add("num:" + m.group().replace(",", "").strip(".-/"))
    for m in re.finditer(r"\b(section|sec\.|s\.|article|art\.|order|rule|regulation)\s*(\d+[a-z]*(?:\(\w+\))*)", t, re.I): facts.add("ref:" + m.group(2).lower())
    for m in re.finditer(r"\b((?:[A-Z][a-zA-Z.]+\s+){0,6}(?:Act|Code|Constitution|Rules))\b", t): facts.add("law:" + re.sub(r"\s+", " ", m.group(1).lower()))
    for m in re.finditer(r"\b([A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,})+)\b", t):
        words = [w for w in m.group(1).split() if w not in _STOPCAP]
        if len(words) >= 2: facts.add("name:" + " ".join(words).lower())
    low = t.lower()
    for w in DECISION_WORDS:
        if re.search(r"\b" + w + r"\b", low): facts.add("dec:" + w)
    return facts
def fact_coverage(ref_facts, text):
    if not ref_facts: return float("nan")
    tf = key_facts(text); low = (text or "").lower(); hit = 0
    for f in ref_facts:
        kind, val = f.split(":", 1)
        if f in tf or (kind in ("num", "ref") and re.search(r"\b" + re.escape(val) + r"\b", low)) or (kind in ("name", "law") and val in low): hit += 1
    return hit / len(ref_facts)
def unsupported_rate(summary, document):
    """share of the summary's numbers and proper names that do NOT occur in the source document: a hallucination proxy"""
    sf = [f for f in key_facts(summary) if f.split(":")[0] in ("num", "name")]
    if not sf: return 0.0
    low = (document or "").lower(); return sum(1 for f in sf if f.split(":", 1)[1] not in low) / len(sf)
def repetition_rate(text, n=3):
    w = rouge_tokens(text); g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 0.0 if not g else 1.0 - len(set(g)) / len(g)
def decision_agree(pred, ref):
    r = {f for f in key_facts(ref) if f.startswith("dec:")}
    if not r: return float("nan")
    return float(bool(r & {f for f in key_facts(pred) if f.startswith("dec:")}))

def score_summaries(preds, refs, docs=None, with_bertscore=True):
    """per-document scores (DataFrame) + means (dict). Never call it with test data: test references are private."""
    assert len(preds) == len(refs)
    r = [rouge_pair(p, g) for p, g in zip(preds, refs)]
    df = pd.DataFrame({"rouge1_f": [a for a, _ in r], "rougeL_f": [b for _, b in r]})
    bs = bertscore_f1(preds, refs) if with_bertscore else None; df["bertscore_f1"] = bs if bs is not None else np.nan
    df["pred_words"] = [len((p or "").split()) for p in preds]; df["ref_words"] = [len((g or "").split()) for g in refs]; df["len_ratio"] = df.pred_words / df.ref_words.clip(lower=1)
    df["fact_coverage"] = [fact_coverage(key_facts(g), p) for p, g in zip(preds, refs)]; df["decision_agree"] = [decision_agree(p, g) for p, g in zip(preds, refs)]
    df["repetition3"] = [repetition_rate(p) for p in preds]; df["empty"] = [float(not (p or "").strip()) for p in preds]
    if docs is not None: df["unsupported"] = [unsupported_rate(p, d) for p, d in zip(preds, docs)]
    means = {k: (float(np.nanmean(df[k])) if df[k].notna().any() else float("nan")) for k in df.columns}
    means.update(n_docs=len(df), rouge_impl=ROUGE_IMPL, bertscore_setting=bertscore_setting() if bs is not None else "not computed")
    return df, means
def sel_metric(m):
    """the selection metric: BERTScore F1 when computed, else the mean of ROUGE-1 and ROUGE-L F1 (stated in every table that uses it)"""
    b = m.get("bertscore_f1", float("nan"))
    return (float(b), "bertscore_f1") if b == b else (0.5 * (m["rouge1_f"] + m["rougeL_f"]), "mean_rouge1_rougeL")
def paired_bootstrap(a, b, n=1000, seed=0):
    """paired bootstrap over documents of mean(a) - mean(b): mean difference, 95% interval, share of resamples where a wins"""
    a, b = np.asarray(a, float), np.asarray(b, float); ok = ~(np.isnan(a) | np.isnan(b)); a, b = a[ok], b[ok]
    if len(a) == 0: return float("nan"), float("nan"), float("nan"), float("nan")
    r = np.random.default_rng(seed); idx = r.integers(0, len(a), (n, len(a))); d = (a[idx] - b[idx]).mean(1)
    return float((a - b).mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d > 0).mean())

# ---- the submission file: Option A, a JSON list [{"id": <id exactly as in the test file>, "summary": "<generated text>"}] ----
def to_py(x): return x.item() if hasattr(x, "item") else x
def write_submission(path, ids, summaries):
    json.dump([{"id": to_py(i), PRED_KEY: str(s)} for i, s in zip(ids, summaries)], open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
def validate_submission(path, ids, forbid_texts=None):
    """one object per test id, the ids preserved in order and type, ONLY the keys id + summary, non-empty generated text, no duplicates; forbid_texts = texts the summary must not equal (e.g. the test documents)"""
    sub = json.load(open(path, encoding="utf-8")); assert isinstance(sub, list), f"{path}: the top level must be a JSON list (Option A)"
    assert len(sub) == len(ids), f"{path}: expected {len(ids)} objects, found {len(sub)}"
    for s, i in zip(sub, ids):
        assert isinstance(s, dict) and set(s) == {"id", PRED_KEY}, f"bad keys {set(s) if isinstance(s, dict) else type(s)}: only 'id' and '{PRED_KEY}' are allowed"
        assert s["id"] == to_py(i) and type(s["id"]) == type(to_py(i)), f"id mismatch or changed type: {s['id']!r} vs {to_py(i)!r}"
        assert isinstance(s[PRED_KEY], str) and s[PRED_KEY].strip(), f"empty or non-string summary for id {s['id']!r}"
    assert len({json.dumps(s["id"]) for s in sub}) == len(sub), "duplicate ids"
    if forbid_texts is not None:
        same = sum(1 for s, d in zip(sub, forbid_texts) if s[PRED_KEY].strip() == (d or "").strip()); assert same == 0, f"{same} summaries are copies of their document"
    return True
print(f"ROUGE implementation: {ROUGE_IMPL} (Porter stemming {'on' if (ROUGE_STEMMER and (ROUGE_IMPL == 'rouge_score' or _PORTER is not None)) else 'off'}) | BERTScore: {'library default for ' + BERTSCORE_LANG + ' (not necessarily the official setting)' if BERTSCORE else 'disabled'}")


In [ ]:
# ================= CELL 4 — LOAD THE OFFICIAL PHASE 6 DATA, VERIFY THE SCHEMA, EXPLORE IT, AND MAKE THE HELD-OUT VALIDATION SPLIT  [RUNS NOW: CPU only, no checkpoints] =================
# Nothing about the data is assumed: the splits, the field names, the row counts, missing values, duplicate ids and lengths are inspected and printed. There is NO public validation split: a reproducible one is cut from train.
ID_CANDS = ("id", "doc_id", "document_id", "ID", "Id", "case_id", "uid", "name")
DOC_CANDS = ("document", "text", "judgment", "judgement", "doc", "source", "input", "full_text", "article", "content")
SUM_CANDS = ("summary", "abstract", "headnote", "headnotes", "target", "output", "highlights", "reference", "gold")
def _records(obj):
    if isinstance(obj, list): return obj
    if isinstance(obj, dict):
        lists = [v for v in obj.values() if isinstance(v, list) and v and isinstance(v[0], dict)]
        if lists: return max(lists, key=len)
        if obj and all(isinstance(v, dict) for v in obj.values()): return [{**v, "__key__": k} for k, v in obj.items()]          # {id: record} layout
    raise ValueError(f"unrecognised JSON layout: top level {type(obj).__name__}")
def read_json_any(p):
    try: return _records(json.load(open(p, encoding="utf-8")))
    except json.JSONDecodeError: return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
def rows_hf():
    from datasets import load_dataset, get_dataset_config_names
    tk = os.environ.get("HF_TOKEN")
    try: cfgs = get_dataset_config_names(HF_DATASET, token=tk)
    except Exception: cfgs = None
    if cfgs is not None: assert HF_CONFIG in cfgs, f"config '{HF_CONFIG}' not found in {HF_DATASET}; available configs: {cfgs}"
    ds = load_dataset(HF_DATASET, HF_CONFIG, token=tk); names = list(ds.keys()); print("Hugging Face splits found:", {k: len(ds[k]) for k in names})
    tr = [n for n in names if "train" in n.lower()]; te = [n for n in names if "test" in n.lower()]; va = [n for n in names if re.search(r"val|dev", n, re.I)]
    assert tr and te, f"expected a train and a test split, found {names}"
    out = {"train": [dict(r) for r in ds[tr[0]]], "test": [dict(r) for r in ds[te[0]]]}
    if va: out["provided_validation"] = [dict(r) for r in ds[va[0]]]
    return out
def rows_files():
    roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else ["/kaggle/input"]; out = {}
    for sp in ("train", "test"):
        c = sorted(p for r in roots for p in glob.glob(f"{r}/**/*.json*", recursive=True) if sp in os.path.basename(p).lower() and "prediction" not in p.lower() and "/p6/" not in p)
        assert c, f"no {sp} json file found under {roots} (expected the Phase_6_Summ/Train/train.json, Test/test.json layout)"
        pref = [p for p in c if re.search(r"phase.?6|summ", p, re.I)]; p = max(pref or c, key=os.path.getsize); out[sp] = read_json_any(p)          # prefer a Phase_6_Summ-like path; print(f"[{sp}] {p}: {len(out[sp])} records")
    return out
def load_raw():
    if DATA_SOURCE == "hf":
        try: return rows_hf()
        except Exception as e:
            print(f"Hugging Face load failed: {type(e).__name__}: {str(e)[:300]}\n-> falling back to local files ({LOCAL_DATA_DIR or '/kaggle/input'})")
    return rows_files()
RAW = load_raw()
def detect(rows, cands, what, override, required=True):
    keys = Counter(k for r in rows[:200] for k in r)
    if override: assert override in keys, f"{what} key '{override}' (set in the config) is not in the data: keys {dict(keys)}"; return override
    for c in cands:
        if c in keys: return c
    if what == "id" and "__key__" in keys: return "__key__"
    if not required: return None
    strs = [k for k in keys if isinstance(rows[0].get(k), str)]
    raise KeyError(f"could not detect the {what} field among keys {dict(keys)} (string fields: {strs}): set {what.upper()}_KEY in the config cell")
ID_F = detect(RAW["train"], ID_CANDS, "id", ID_KEY, required=False); DOC_F = detect(RAW["train"], DOC_CANDS, "doc", DOC_KEY); SUM_F = detect(RAW["train"], SUM_CANDS, "sum", SUM_KEY)
TEST_ID_F = detect(RAW["test"], ID_CANDS, "id", ID_KEY, required=False); TEST_DOC_F = detect(RAW["test"], DOC_CANDS, "doc", DOC_KEY)
print(f"\nSCHEMA: train keys {sorted(Counter(k for r in RAW['train'][:200] for k in r))} | test keys {sorted(Counter(k for r in RAW['test'][:200] for k in r))}")
print(f"  using id='{ID_F}', document='{DOC_F}', summary='{SUM_F}' (train) | id='{TEST_ID_F}', document='{TEST_DOC_F}' (test)")
assert TEST_ID_F is not None, "the test records have no id field: the submission needs the original ids; set ID_KEY in the config cell"
if ID_F is None: print("WARNING: the training records have no id field: positional ids 'train_<n>' are used (they never reach a submission)")
if SUM_F in Counter(k for r in RAW["test"][:200] for k in r): print(f"[test] NOTE: the test records contain a '{SUM_F}' field. Test references are private, so it is IGNORED and never read.")
if "provided_validation" in RAW: print(f"NOTE: the dataset also has a split named like validation ({len(RAW['provided_validation'])} rows). The specification says there is none; it is NOT used, and the held-out split below is made from train.")

def clean_text(x): return "" if x is None else (x if isinstance(x, str) else " ".join(map(str, x)) if isinstance(x, list) else str(x))
def build(rows, idf, docf, sumf, name):
    ids = [to_py(r.get(idf)) if idf else f"{name}_{i}" for i, r in enumerate(rows)]; docs = [clean_text(r.get(docf)) for r in rows]
    sums = [clean_text(r.get(sumf)) for r in rows] if sumf else None
    return dict(ids=ids, docs=docs, sums=sums)
TRAIN_ALL = build(RAW["train"], ID_F, DOC_F, SUM_F, "train"); TEST = build(RAW["test"], TEST_ID_F, TEST_DOC_F, None, "test"); del RAW; gc.collect()

# ---- quality checks: missing / empty values, duplicate ids, duplicate documents ----
def wc(t): return len(t.split())
miss = dict(train_empty_doc=sum(not d.strip() for d in TRAIN_ALL["docs"]), train_empty_summary=sum(not s.strip() for s in TRAIN_ALL["sums"]), test_empty_doc=sum(not d.strip() for d in TEST["docs"]),
            train_missing_id=sum(i is None for i in TRAIN_ALL["ids"]), test_missing_id=sum(i is None for i in TEST["ids"]))
dup_tr = [k for k, v in Counter(map(str, TRAIN_ALL["ids"])).items() if v > 1]; dup_te = [k for k, v in Counter(map(str, TEST["ids"])).items() if v > 1]
dup_docs = len(TRAIN_ALL["docs"]) - len(set(TRAIN_ALL["docs"])); overlap = set(TRAIN_ALL["docs"]) & set(TEST["docs"])
print(f"\nROWS: train {len(TRAIN_ALL['ids'])} | test {len(TEST['ids'])}\nMISSING / EMPTY: {miss}\nDUPLICATES: train ids {len(dup_tr)} {dup_tr[:5]} | test ids {len(dup_te)} {dup_te[:5]} | identical train documents {dup_docs} | test documents identical to a train document {len(overlap)}")
assert not dup_te and not miss["test_missing_id"], "duplicate or missing test ids: the submission cannot be one-object-per-id; inspect the test file"
keep = [i for i in range(len(TRAIN_ALL["ids"])) if TRAIN_ALL["docs"][i].strip() and TRAIN_ALL["sums"][i].strip()]
if len(keep) < len(TRAIN_ALL["ids"]): print(f"dropping {len(TRAIN_ALL['ids']) - len(keep)} training records with an empty document or summary")
if dup_tr:                                                                    # keep the first record of a duplicated id (identical ids would otherwise break the split)
    seen = set(); keep = [i for i in keep if not (str(TRAIN_ALL["ids"][i]) in seen or seen.add(str(TRAIN_ALL["ids"][i])))]
TRAIN_ALL = {k: [v[i] for i in keep] for k, v in TRAIN_ALL.items()}
if any(not d.strip() for d in TEST["docs"]): print("WARNING: some test documents are empty: they will still receive a (generic) generated summary so that every id has one")

# ---- the held-out validation split: deterministic (a hash of SPLIT_SEED + document text, so identical documents always fall on the same side and every notebook recomputes the same split) ----
def hkey(s): return hashlib.sha1(f"{SPLIT_SEED}:{s}".encode("utf-8")).hexdigest()
N = len(TRAIN_ALL["ids"]); n_val = int(min(VAL_MAX, max(VAL_MIN, round(VAL_FRACTION * N)), N // 2)); assert n_val >= 1, f"only {N} usable training records: too few to hold out a validation split"
groups = defaultdict(list)
for i, d in enumerate(TRAIN_ALL["docs"]): groups[hkey(d)].append(i)
val_idx = []
for g in sorted(groups):
    if len(val_idx) >= n_val: break
    val_idx += groups[g]
val_set = set(val_idx); tr_idx = [i for i in range(N) if i not in val_set]
TRAIN = {k: [v[i] for i in tr_idx] for k, v in TRAIN_ALL.items()}; VAL = {k: [v[i] for i in val_idx] for k, v in TRAIN_ALL.items()}
assert not (set(map(str, TRAIN["ids"])) & set(map(str, VAL["ids"]))) and not (set(TRAIN["docs"]) & set(VAL["docs"])), "train / validation overlap"
split_info = dict(seed=SPLIT_SEED, n_train=len(TRAIN["ids"]), n_val=len(VAL["ids"]), val_ids_sha1=hashlib.sha1(json.dumps(sorted(map(str, VAL["ids"]))).encode()).hexdigest(), val_ids=[str(i) for i in VAL["ids"]])
sp = WORK / "split.json"
if sp.exists():
    old = json.load(open(sp))
    if old.get("val_ids_sha1") != split_info["val_ids_sha1"]: print("WARNING: the split differs from the one restored from an earlier part (different data or settings?); the restored validation scores are NOT comparable with this session's")
json.dump(split_info, open(sp, "w"), indent=0)
EVAL_IDX = sorted(range(len(VAL["ids"])), key=lambda i: hkey("eval" + str(VAL["ids"][i])))[:VAL_EVAL_CAP]          # the fixed validation subset every experiment is generated and scored on
EVAL_SIG = hashlib.sha1(json.dumps([str(VAL["ids"][i]) for i in EVAL_IDX]).encode()).hexdigest()[:10]
print(f"\nHELD-OUT VALIDATION SPLIT (made from train; references kept apart from training): train {len(TRAIN['ids'])} | validation {len(VAL['ids'])} (fraction {VAL_FRACTION}, min {VAL_MIN}, max {VAL_MAX}, seed {SPLIT_SEED})"
      f" | scored subset: {len(EVAL_IDX)} documents, signature {EVAL_SIG} (identical in every notebook)")

# ---- exploration: lengths in words (token lengths follow after tokenisation) ----
def stats(xs): xs = np.asarray(xs); return dict(n=len(xs), mean=int(xs.mean()) if len(xs) else 0, p10=int(np.percentile(xs, 10)) if len(xs) else 0, p50=int(np.percentile(xs, 50)) if len(xs) else 0, p90=int(np.percentile(xs, 90)) if len(xs) else 0, max=int(xs.max()) if len(xs) else 0)
rows = [dict(what=f"{n} documents", **stats([wc(d) for d in D["docs"]])) for n, D in (("train", TRAIN), ("validation", VAL), ("test", TEST))] + [dict(what=f"{n} summaries", **stats([wc(s) for s in D["sums"]])) for n, D in (("train", TRAIN), ("validation", VAL))]
print("\nLENGTHS IN WORDS:"); print(pd.DataFrame(rows).to_string(index=False))
comp = np.array([wc(s) / max(1, wc(d)) for d, s in zip(TRAIN["docs"], TRAIN["sums"])]); print(f"summary / document length ratio (train): median {np.median(comp):.3f}, p10 {np.percentile(comp, 10):.3f}, p90 {np.percentile(comp, 90):.3f}")
novel = np.mean([1 - len(set(rouge_tokens(s)) & set(rouge_tokens(d))) / max(1, len(set(rouge_tokens(s)))) for d, s in list(zip(TRAIN["docs"], TRAIN["sums"]))[:500]])
print(f"share of summary word types NOT in the document (abstractiveness, first 500 train pairs): {novel:.3f}")
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.4)); ax[0].hist(np.log10([wc(d) + 1 for d in TRAIN["docs"]]), bins=40); ax[0].set_title("log10 words per document (train)")
    ax[1].hist([wc(s) for s in TRAIN["sums"]], bins=40); ax[1].set_title("words per summary (train)"); ax[2].hist(comp, bins=40); ax[2].set_title("summary / document length")
    plt.tight_layout(); plt.savefig(WORK / "data_overview.png", dpi=100); plt.close(); print("saved", WORK / "data_overview.png")
except Exception as e: print("plot skipped:", type(e).__name__)


In [ ]:
# ================= CELL 5 — TOKENISATION AND PREPROCESSING  [NEEDS the Phase 1 tokenizer file legal_bpe_32k_mask.json; CPU only] =================
# Documents are split into sentences (a regex suited to judgments) and every sentence is tokenised once; all long-document strategies (head / tail / head+tail / importance selection / section-aware / chunks) are
# built from these sentence tokens, so a strategy only decides WHICH tokens a model reads. Summaries are tokenised whole. Special tokens are looked up by name, never assumed.
from tokenizers import Tokenizer
DOC_CHAR_CAP = 300_000                                   # characters kept per document (first 2/3 + last 1/3 of the cap for the rare giant document), only to bound memory
tpaths = [TOKENIZER_PATH] if TOKENIZER_PATH else (sorted(glob.glob("/kaggle/input/**/legal_bpe_32k_mask.json", recursive=True)) or sorted(glob.glob("/kaggle/input/**/legal_bpe_32k*.json", recursive=True)) or sorted(glob.glob("/kaggle/working/**/legal_bpe_32k*.json", recursive=True)))
assert tpaths and os.path.exists(tpaths[0]), "tokenizer not found: attach a Phase 1 dataset (it holds legal_bpe_32k_mask.json) or set TOKENIZER_PATH in the config cell"
tok = Tokenizer.from_file(tpaths[0]); VOCAB = tok.get_vocab_size()
PAD, UNK, BOS, EOS, MASK = (tok.token_to_id(t) for t in ("[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"))
assert None not in (PAD, BOS, EOS), f"special tokens missing from the tokenizer: PAD={PAD} BOS={BOS} EOS={EOS}"
SPECIAL_IDS = {x for x in (PAD, UNK, BOS, EOS, MASK) if x is not None}
def enc(text): return tok.encode(text, add_special_tokens=False).ids
def detok(ids):
    ids = [int(i) for i in ids if int(i) not in SPECIAL_IDS]; t = tok.decode(ids, skip_special_tokens=True) if ids else ""
    if "Ġ" in t or "Ċ" in t: t = t.replace("Ġ", " ").replace("Ċ", "\n")                         # a tokenizer file without a byte-level decoder
    return re.sub(r"[ \t]+", " ", t).strip()
_rt = "The appellant was convicted under Section 302 IPC on 12.03.1998; the appeal is dismissed."
print(f"tokenizer: {tpaths[0]} | vocabulary {VOCAB} | PAD={PAD} UNK={UNK} BOS={BOS} EOS={EOS} MASK={MASK} | round trip OK: {detok(enc(_rt)) == _rt} -> {detok(enc(_rt))!r}")
SEP_IDS = enc("\n\nSummary:")                                                                  # the separator between document and summary for the decoder-only (GPT-2) format
ELLIPSIS_IDS = enc(" ... ")

SENT_RE = re.compile(r"(?<=[.?!;:])\s+(?=[\"'(\[]?[A-Z0-9])|\n{2,}|\n(?=\s*(?:\d+[.)]|\(?[ivxIVX]+\)|[A-Z][A-Z ]{3,}:?\s*$))")
def sentences(text):
    text = text if len(text) <= DOC_CHAR_CAP else text[: 2 * DOC_CHAR_CAP // 3] + "\n\n" + text[-DOC_CHAR_CAP // 3:]
    return [s.strip() for s in SENT_RE.split(text.replace("\r", "")) if s and s.strip()]
STOP = set("the a an of to in and or for on at by with from that this these those is are was were be been being it its as which who whom whose not no but if than then there their they he she his her him we our you your i shall may can could would should will has have had do does did said such any all also upon under into".split())
CUES = ("held", "hold", "issue", "question", "contend", "submit", "argu", "fact", "appeal", "petition", "prosecution", "accused", "reason", "conclu", "therefore", "accordingly", "order", "judgment", "finding", "relief", "convict", "acquit", "dismiss", "allow", "quash", "set aside", "uphold", "sentence")
def sentence_scores(sents):
    """importance of each sentence WITHIN its own document (no corpus statistics, no learned weights): content-word salience + legal cue words + numbers / statute references + position"""
    toks = [[w for w in rouge_tokens(s) if w not in STOP and len(w) > 2] for s in sents]; tf = Counter(w for t in toks for w in t); n = len(sents); sc = np.zeros(n)
    for k, (s, t) in enumerate(zip(sents, toks)):
        if not t: continue
        low = s.lower(); sal = sum(math.log1p(tf[w]) for w in set(t)) / math.sqrt(len(t) + 3)
        sc[k] = sal + 0.6 * any(c in low for c in CUES) + 0.3 * bool(re.search(r"\b(section|article|act)\b|\d", low)) + 0.4 * (k < 4) + 0.4 * (k >= n - 4) - 0.5 * (len(t) < 4)
    return sc

def tokenise_split(D, name):
    sig = hashlib.sha1(json.dumps([str(x) for x in D["ids"]] + [len(d) for d in D["docs"]]).encode()).hexdigest()[:10]
    cache = WORK / "cache" / f"tok_{name}_{len(D['ids'])}_{sig}_{VOCAB}.pkl"
    if cache.exists():
        D.update(pickle.load(open(cache, "rb"))); print(f"[{name}] loaded the tokenisation cache"); return
    t0 = time.time(); sents = [sentences(d) for d in D["docs"]]; flat = [s for ss in sents for s in ss]; ids = []
    for i in range(0, len(flat), 4096): ids += [e.ids for e in tok.encode_batch([" " + s for s in flat[i:i + 4096]], add_special_tokens=False)]
    out, p = dict(sent_toks=[], sent_scores=[], doc_len=[]), 0
    for ss in sents:
        st = [np.asarray(x, dtype=np.int32) for x in ids[p:p + len(ss)]]; p += len(ss)
        out["sent_toks"].append(st); out["sent_scores"].append(sentence_scores(ss).astype(np.float32)); out["doc_len"].append(int(sum(len(x) for x in st)))
    out["sents"] = sents
    if D.get("sums") is not None: out["sum_toks"] = [np.asarray(e.ids, dtype=np.int32) for e in tok.encode_batch(D["sums"], add_special_tokens=False)]
    D.update(out); pickle.dump(out, open(cache, "wb"), protocol=4); print(f"[{name}] tokenised {len(D['ids'])} documents ({len(flat)} sentences) in {time.time() - t0:.0f}s")
for name, D in (("train", TRAIN), ("val", VAL), ("test", TEST)): tokenise_split(D, name)
def tstats(xs): xs = np.asarray(xs); return dict(p10=int(np.percentile(xs, 10)), p50=int(np.percentile(xs, 50)), p75=int(np.percentile(xs, 75)), p90=int(np.percentile(xs, 90)), max=int(xs.max()), **{f">{b}": f"{np.mean(xs > b):.0%}" for b in (512, 1024, 2048, 4096)})
print("\nLENGTHS IN TOKENS (Phase 1 tokenizer):"); print(pd.DataFrame([dict(what=f"{n} documents", **tstats(D["doc_len"])) for n, D in (("train", TRAIN), ("validation", VAL), ("test", TEST))]
                                                                   + [dict(what=f"{n} summaries", **tstats([len(s) for s in D["sum_toks"]])) for n, D in (("train", TRAIN), ("validation", VAL))]).to_string(index=False))
SUM_P75, SUM_P90 = int(np.percentile([len(s) for s in TRAIN["sum_toks"]], 75)), int(np.percentile([len(s) for s in TRAIN["sum_toks"]], 90))
LONG_THRESHOLD = int(np.median(VAL["doc_len"]))                                               # 'short' vs 'long' documents in every per-length report = below / above the validation median
print(f"summary tokens p75 {SUM_P75}, p90 {SUM_P90} | short/long document threshold (validation median) {LONG_THRESHOLD} tokens")

# ---- long-document input strategies: each returns at most `budget` token ids of one document ----
def _concat(parts): return np.concatenate(parts).astype(np.int64) if parts else np.zeros(0, dtype=np.int64)
def _pick(st, order, budget, scores=None, mmr=0.6):
    chosen, used, seen = [], 0, []
    for k in order:
        L = len(st[k])
        if L == 0 or used + L > budget: continue
        if scores is not None and mmr < 1.0:
            s = set(st[k].tolist())
            if any(len(s & t) / max(1, len(s)) > mmr for t in seen): continue                   # redundancy control: skip a sentence that mostly repeats a chosen one
            seen.append(s)
        chosen.append(k); used += L
        if used >= budget - 4: break
    return sorted(chosen)
def build_input(D, i, strategy, budget):
    if "@" in strategy: strategy, cap = strategy.split("@"); budget = min(budget, int(cap))       # 'head@128' = the same strategy with a shorter input (truncation-length study)
    st, sc = D["sent_toks"][i], D["sent_scores"][i]; full = _concat(st)
    if len(full) <= budget or strategy == "head": return full[:budget]
    if strategy == "tail": return full[-budget:]
    if strategy == "head_tail": h = (budget - len(ELLIPSIS_IDS)) // 2; return _concat([full[:h], np.asarray(ELLIPSIS_IDS), full[-(budget - len(ELLIPSIS_IDS) - h):]])
    if strategy == "importance": return _concat([st[k] for k in _pick(st, np.argsort(-sc, kind="stable"), budget, sc)])
    if strategy == "section":                                                                 # beginning (facts) 40%, middle (arguments) 20%, end (reasoning + decision) 40%; importance-ranked inside each zone, document order kept
        n = len(st); zones = [(0, max(1, int(0.3 * n)), 0.4), (max(1, int(0.3 * n)), max(1, int(0.7 * n)), 0.2), (max(1, int(0.7 * n)), n, 0.4)]; keep = []
        for a, b, f in zones:
            idx = np.arange(a, b); keep += [idx[k] for k in _pick([st[j] for j in idx], np.argsort(-sc[a:b], kind="stable"), int(budget * f), sc[a:b])]
        return _concat([st[k] for k in sorted(keep)])[:budget]
    if strategy == "oracle":                                                                  # UPPER BOUND for analysis only: needs the reference summary, never usable on test documents
        ref = set(rouge_tokens(D["sums"][i])); r = np.array([len(set(rouge_tokens(s)) & ref) / math.sqrt(len(rouge_tokens(s)) + 3) for s in D["sents"][i]]); return _concat([st[k] for k in _pick(st, np.argsort(-r, kind="stable"), budget)])
    raise ValueError(strategy)
def chunk_windows(D, i, budget, overlap=0.25, max_chunks=6):
    """overlapping sliding windows over the whole document (stride = budget * (1 - overlap)); at most max_chunks, spread evenly when there are more"""
    full = _concat(D["sent_toks"][i])
    if len(full) <= budget: return [full]
    stride = max(1, int(budget * (1 - overlap))); starts = list(range(0, max(1, len(full) - budget) + 1, stride))
    if starts[-1] + budget < len(full): starts.append(len(full) - budget)
    if len(starts) > max_chunks: starts = [starts[int(round(k))] for k in np.linspace(0, len(starts) - 1, max_chunks)]
    return [full[s:s + budget] for s in starts]
def lead_text(D, i, words):
    out, n = [], 0
    for s in D["sents"][i]:
        out.append(s); n += len(s.split())
        if n >= words: break
    return " ".join(out)


In [ ]:
# ================= CELL 6 — PHASE 1 CHECKPOINT DISCOVERY + GENERATION-CAPABLE WRAPPERS  [NEEDS the Phase 1 checkpoints for the models you use; set CKPT_PATHS in the config cell or attach the datasets]. The old 3,000-step GPT-2 is never used. =================
# How each Phase 1 architecture generates a summary (no outside weights anywhere; every weight below starts from OUR Phase 1 checkpoints unless it is listed as new):
#   gpt2    decoder-only: [BOS] document [SEP='\n\nSummary:'] summary [EOS]; loss on the summary tokens only. 512 learned positions = document + summary budget (positions are NOT extended: that would add untrained weights).
#   t5      native encoder-decoder (relative positions, so the encoder may read more than 512 tokens). fp32 by default (fp16 was unstable in Phases 4-5).
#   encdec  Phase 1 ModernBERT ENCODER + Phase 1 GPT-2 DECODER (both 768-wide, same tokenizer) joined by Hugging Face EncoderDecoderModel; NEW weights = the cross-attention blocks
#           (crossattention.* and ln_cross_attn.* in every GPT-2 layer) and nothing else. Counted as the ModernBERT architecture (user decision).
#   bilstm  sequence-to-sequence with attention: encoder = the Phase 1 bidirectional LSTMs, decoder = a COPY of the Phase 1 forward LSTM + the Phase 1 output head; NEW weights = a Luong attention
#           matrix and a zero-initialised combination layer (so at initialisation the decoder is exactly the Phase 1 forward language model). Greedy decoding only. Expect weak output: the Phase 1 BiLSTM is barely trained.
ARCH_LOADERS_SRC = r'''"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.
   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)
   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""
import torch, torch.nn as nn
from types import SimpleNamespace
from transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration

class BiLSTMLM(nn.Module):
    def __init__(self, vocab, emb=512, hidden=1024, layers=3):
        super().__init__()
        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers
        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)
        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))
    def hidden_states(self, input_ids, attention_mask=None):
        x = input_ids; B, T = x.shape
        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)
        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))
        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))
        return torch.cat([hf, hb], dim=-1)

class Backbone(nn.Module):
    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""
    def __init__(self, family, inner, embed):
        super().__init__(); self.family, self.inner, self._embed = family, inner, embed
    def get_input_embeddings(self): return self._embed
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):
        f = self.family
        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state
        else:
            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"
            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)
            else: raise ValueError(f)
        return SimpleNamespace(last_hidden_state=h)

CAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}
# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT's q,k,v are ONE fused linear, 'Wqkv' (rows: q | k | v).
ADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}

def build_full(mc):
    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]
    if f == "gpt2":
        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),
                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))
    if f == "modernbert":
        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],
                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,
                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))
    if f == "t5":
        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],
                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))
    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])
    raise ValueError(f)

def load_backbone(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)
    f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f
'''
al_paths = sorted(glob.glob("/kaggle/input/**/arch_loaders.py", recursive=True)) or sorted(glob.glob("/kaggle/working/**/arch_loaders.py", recursive=True))
if not al_paths:                                                                  # self-contained: the loader module is embedded in this notebook
    (WORK / "arch_loaders.py").write_text(ARCH_LOADERS_SRC); al_paths = [str(WORK / "arch_loaders.py")]; print("arch_loaders.py not found among the inputs: using the copy embedded in this notebook")
sys.path.insert(0, str(Path(al_paths[0]).parent)); import arch_loaders as AL
from transformers import EncoderDecoderModel, GPT2Config, GPT2LMHeadModel

MODEL_PATHS = {}
def discover_models():
    found = {}
    if not DISCOVER_MODELS: return found
    for p in sorted(glob.glob("/kaggle/input/**/final_model.pt", recursive=True)):          # only the NEW Phase 1 models: the old checkpoint_latest.pt (3,000-step GPT-2) is never considered
        if os.path.getsize(p) < 20e6: continue
        try: ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: continue
        if not isinstance(ck, dict) or "model_config" not in ck or "model_state_dict" not in ck: continue
        fam = ck["model_config"].get("family")
        if fam is None: continue
        key = "gpt2_new" if fam == "gpt2" else fam
        if key not in found: found[key] = p
    return found
def refresh_models(verbose=True):
    new = {k: v for k, v in discover_models().items() if k not in MODEL_PATHS}; MODEL_PATHS.update(new)
    if verbose and new: print("newly found Phase 1 models:", new)
    return sorted(MODEL_PATHS)
MODEL_PATHS.update(discover_models()); MODEL_PATHS.update({k: v for k, v in CKPT_PATHS.items() if v})        # CKPT_PATHS from the config cell wins
ARCH_NEEDS = {"gpt2": ("gpt2_new",), "t5": ("t5",), "encdec": ("modernbert", "gpt2_new"), "bilstm": ("bilstm",)}
ARCH_FAMILY = {"gpt2": "gpt2", "t5": "t5", "encdec": "modernbert", "bilstm": "bilstm"}         # the Phase 1 architecture an experiment counts for in the rule check
def arch_available(arch): return all(k in MODEL_PATHS for k in ARCH_NEEDS[arch])
print("Phase 1 models found:", MODEL_PATHS, "\n  missing:", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS] or "none",
      "\n  architectures that can run:", [a for a in ARCH_NEEDS if arch_available(a)], "| cannot run:", [a for a in ARCH_NEEDS if not arch_available(a)])

def load_full(key):
    ck = torch.load(MODEL_PATHS[key], map_location="cpu", weights_only=False); mc = dict(ck["model_config"]); full = AL.build_full(mc)
    full.load_state_dict(ck["model_state_dict"], strict=True); assert mc.get("vocab_size") == VOCAB, f"{key}: vocabulary {mc.get('vocab_size')} != tokenizer {VOCAB}"; del ck; return full, mc
def set_special_ids(cfg, start=None):
    for k, v in (("bos_token_id", BOS), ("eos_token_id", EOS), ("pad_token_id", PAD)): setattr(cfg, k, v)
    if start is not None: cfg.decoder_start_token_id = start
def truncate_gpt2(lm, n):
    lm.transformer.h = nn.ModuleList(list(lm.transformer.h)[:n]); lm.config.n_layer = n; return lm
def truncate_t5(m, n):
    m.encoder.block = nn.ModuleList(list(m.encoder.block)[:n]); m.decoder.block = nn.ModuleList(list(m.decoder.block)[:n]); m.config.num_layers = n; m.config.num_decoder_layers = n
    m.encoder.config.num_layers = n; m.decoder.config.num_layers = n; return m                  # block 0 holds the relative-position bias, so keeping the first n blocks is valid

def gen_kwargs(g, seq2seq):
    kw = dict(max_new_tokens=int(g["max_new"]), min_new_tokens=int(g.get("min_new", 1)), num_beams=int(g.get("beams", 1)), no_repeat_ngram_size=int(g.get("no_repeat", 0)),
              repetition_penalty=float(g.get("rep_pen", 1.0)), eos_token_id=EOS, pad_token_id=PAD, do_sample=bool(g.get("sample", False)))
    if kw["num_beams"] > 1: kw.update(length_penalty=float(g.get("len_pen", 1.0)), early_stopping=True)
    if kw["do_sample"]: kw.update(top_p=float(g.get("top_p", 0.9)), temperature=float(g.get("temperature", 0.7)))
    return kw
def _pad_right(seqs, val):
    L = max(1, max(len(s) for s in seqs)); out = torch.full((len(seqs), L), val, dtype=torch.long)
    for b, s in enumerate(seqs): out[b, :len(s)] = torch.as_tensor(np.asarray(s, dtype=np.int64))
    return out
def _nonempty(s): return s if len(s) else np.asarray(enc("."), dtype=np.int64)

class DecOnlySum(nn.Module):
    """GPT-2 decoder-only summariser (optional prompt tuning: trained virtual-token embeddings prepended to every input)"""
    arch, seq2seq = "gpt2", False
    def __init__(self, lm):
        super().__init__(); self.lm, self.prompt = lm, None; set_special_ids(lm.config); set_special_ids(lm.generation_config); self.ctx = lm.config.n_positions
    def n_virtual(self): return 0 if self.prompt is None else self.prompt.shape[0]
    def enable_prompt(self, n, seed=0):
        g = torch.Generator().manual_seed(seed); ids = torch.randint(10, VOCAB - 1, (n,), generator=g)          # initialised from the embeddings of random vocabulary tokens (Lester et al. style)
        self.prompt = nn.Parameter(self.lm.transformer.wte.weight[ids].detach().clone().float())
    def make_batch(self, srcs, tgts):
        seqs, labs = [], []
        for s, t in zip(srcs, tgts):
            p = [BOS] + list(_nonempty(s)) + SEP_IDS; y = list(t) + [EOS]; seqs.append(p + y); labs.append([-100] * len(p) + y)
        ids = _pad_right(seqs, PAD); lab = _pad_right(labs, -100); mask = (lab != -100) | (torch.arange(ids.shape[1])[None, :] < torch.tensor([len(s) for s in seqs])[:, None])
        return dict(input_ids=ids, attention_mask=mask.long(), labels=lab)
    def _embeds(self, ids, mask):
        e = self.lm.transformer.wte(ids); B = ids.shape[0]
        e = torch.cat([self.prompt.to(e.dtype)[None].expand(B, -1, -1), e], 1); return e, torch.cat([torch.ones(B, self.n_virtual(), dtype=mask.dtype, device=mask.device), mask], 1)
    def forward_logits(self, b):
        ids, mask, lab = b["input_ids"], b["attention_mask"], b["labels"]
        if self.prompt is None: lg = self.lm(input_ids=ids, attention_mask=mask, use_cache=False).logits
        else: e, m = self._embeds(ids, mask); lg = self.lm(inputs_embeds=e, attention_mask=m, use_cache=False).logits[:, self.n_virtual():]
        return lg[:, :-1], lab[:, 1:]                                                     # logits at position t predict token t+1
    @torch.no_grad()
    def generate_ids(self, srcs, g):
        prompts = [[BOS] + list(_nonempty(s)) + SEP_IDS for s in srcs]; L = max(len(p) for p in prompts)
        ids = torch.full((len(prompts), L), PAD, dtype=torch.long); mask = torch.zeros_like(ids)
        for b, p in enumerate(prompts): ids[b, L - len(p):] = torch.tensor(p); mask[b, L - len(p):] = 1          # LEFT padding for generation
        ids, mask = ids.to(device), mask.to(device); room = self.ctx - L - self.n_virtual(); kw = gen_kwargs({**g, "max_new": max(1, min(int(g["max_new"]), room))}, False)
        kw["min_new_tokens"] = min(kw["min_new_tokens"], kw["max_new_tokens"])
        if self.prompt is None: out = self.lm.generate(input_ids=ids, attention_mask=mask, **kw); new = out[:, L:]
        else:
            e, m = self._embeds(ids, mask); out = self.lm.generate(inputs_embeds=e, attention_mask=m, **kw); new = out if out.shape[1] <= kw["max_new_tokens"] + 1 else out[:, e.shape[1]:]
        return [[int(x) for x in row if int(x) not in (PAD,)] for row in new.cpu().tolist()]

class Seq2SeqSum(nn.Module):
    """T5 or ModernBERT-encoder + GPT-2-decoder (Hugging Face EncoderDecoderModel)"""
    seq2seq = True
    def __init__(self, model, arch):
        super().__init__(); self.model, self.arch = model, arch
    def make_batch(self, srcs, tgts):
        src = [list(_nonempty(s)) + ([EOS] if self.arch == "t5" else []) for s in srcs]; ids = _pad_right(src, PAD); mask = (torch.arange(ids.shape[1])[None, :] < torch.tensor([len(s) for s in src])[:, None]).long()
        return dict(input_ids=ids, attention_mask=mask, labels=_pad_right([list(t) + [EOS] for t in tgts], -100))
    def forward_logits(self, b):
        out = self.model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], labels=b["labels"], use_cache=False); return out.logits, b["labels"]
    @torch.no_grad()
    def generate_ids(self, srcs, g):
        b = self.make_batch(srcs, [[EOS]] * len(srcs)); kw = gen_kwargs(g, True); kw["decoder_start_token_id"] = getattr(self.model.config, "decoder_start_token_id", None) or self.model.generation_config.decoder_start_token_id
        out = self.model.generate(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device), **kw)
        return [[int(x) for x in row] for row in out.cpu().tolist()]

class BiLSTMSum(nn.Module):
    """Phase 1 BiLSTM as a sequence-to-sequence model with attention (greedy decoding)"""
    arch, seq2seq = "bilstm", True
    def __init__(self, lm):
        super().__init__(); self.lm = lm; H = lm.hidden; self.dec = copy.deepcopy(lm.fwd)
        self.att = nn.Linear(H, 2 * H, bias=False); nn.init.normal_(self.att.weight, std=0.02)       # NEW
        self.comb = nn.Linear(3 * H, H); nn.init.zeros_(self.comb.weight); nn.init.zeros_(self.comb.bias)   # NEW, zero-initialised
    def head(self, h): return self.lm.proj_f(self.lm.norm_f(h)) @ self.lm.emb.weight.t() + self.lm.out_bias   # ASSUMED Phase 1 output head (checked by the compatibility cell's language-model loss)
    def make_batch(self, srcs, tgts):
        src = [list(_nonempty(s)) for s in srcs]; ids = _pad_right(src, PAD); mask = (torch.arange(ids.shape[1])[None, :] < torch.tensor([len(s) for s in src])[:, None]).long()
        return dict(input_ids=ids, attention_mask=mask, dec_in=_pad_right([[BOS] + list(t) for t in tgts], PAD), labels=_pad_right([list(t) + [EOS] for t in tgts], -100))
    def _attend(self, d, enc, emask):
        a = torch.bmm(self.att(d), enc.transpose(1, 2)).float().masked_fill(~emask.bool()[:, None, :], -1e4).softmax(-1).to(enc.dtype)
        return d + torch.tanh(self.comb(torch.cat([d, torch.bmm(a, enc)], -1)))
    def forward_logits(self, b):
        enc = self.lm.hidden_states(b["input_ids"], b["attention_mask"]); d, _ = self.dec(self.lm.emb(b["dec_in"])); return self.head(self._attend(d, enc, b["attention_mask"])), b["labels"]
    @torch.no_grad()
    def generate_ids(self, srcs, g):
        b = self.make_batch(srcs, [[]] * len(srcs)); ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device); enc = self.lm.hidden_states(ids, mask)
        B = ids.shape[0]; cur = torch.full((B, 1), BOS, dtype=torch.long, device=device); state = None; outs = [[] for _ in range(B)]; done = torch.zeros(B, dtype=torch.bool, device=device); n = int(g.get("no_repeat", 0))
        for step in range(int(g["max_new"])):
            d, state = self.dec(self.lm.emb(cur), state); lg = self.head(self._attend(d, enc, mask))[:, -1].float()
            if step < int(g.get("min_new", 1)): lg[:, EOS] = -1e9
            if n > 1:
                for r in range(B):
                    seq = outs[r]
                    if len(seq) >= n - 1:
                        pre = tuple(seq[-(n - 1):]); ban = [seq[k + n - 1] for k in range(len(seq) - n + 1) if tuple(seq[k:k + n - 1]) == pre]
                        if ban: lg[r, ban] = -1e9
            if float(g.get("rep_pen", 1.0)) != 1.0:
                for r in range(B):
                    if outs[r]: u = torch.tensor(sorted(set(outs[r])), device=device); v = lg[r, u]; lg[r, u] = torch.where(v > 0, v / g["rep_pen"], v * g["rep_pen"])
            nxt = lg.argmax(-1); nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
            for r in range(B):
                if not done[r] and int(nxt[r]) != EOS: outs[r].append(int(nxt[r]))
            done |= nxt == EOS; cur = nxt[:, None]
            if bool(done.all()): break
        return outs

def make_summarizer(arch, n_layers=None, verbose=True):
    """build a summariser from the Phase 1 checkpoints. n_layers = keep only the first n layers (a smaller distillation student, same Phase 1 weights)"""
    assert arch_available(arch), f"architecture '{arch}' needs Phase 1 checkpoints {ARCH_NEEDS[arch]}; found {sorted(MODEL_PATHS)}"
    if arch == "gpt2":
        lm, mc = load_full("gpt2_new")
        if n_layers: truncate_gpt2(lm, n_layers)
        s = DecOnlySum(lm)
    elif arch == "t5":
        m, mc = load_full("t5")
        if n_layers: truncate_t5(m, n_layers)
        start = getattr(m.config, "decoder_start_token_id", None)                    # transformers 5 keeps it only on generation_config
        if start is None: start = getattr(m.generation_config, "decoder_start_token_id", None)
        if start is None or start >= VOCAB: start = PAD
        set_special_ids(m.config, start); set_special_ids(m.generation_config, start)
        if verbose and start != PAD: print(f"NOTE: T5 decoder_start_token_id = {start} (PAD = {PAD}); kept as in the Phase 1 config")
        s = Seq2SeqSum(m, "t5")
    elif arch == "encdec":
        enc_full, emc = load_full("modernbert"); dec_lm, dmc = load_full("gpt2_new"); encoder = enc_full.model
        assert emc["hidden_size"] == dmc["hidden_size"], f"ModernBERT width {emc['hidden_size']} != GPT-2 width {dmc['hidden_size']}: the encoder-decoder needs a projection (not built)"
        cfg = copy.deepcopy(dec_lm.config); cfg.add_cross_attention = True; cfg.is_decoder = True; dec = GPT2LMHeadModel(cfg)
        missing, unexpected = dec.load_state_dict(dec_lm.state_dict(), strict=False); assert not unexpected, unexpected
        new_ok = all(("crossattention" in k or "ln_cross_attn" in k) for k in missing); assert new_ok, f"unexpected missing decoder weights: {[k for k in missing if 'cross' not in k][:5]}"
        if n_layers: truncate_gpt2(dec, n_layers)
        m = EncoderDecoderModel(encoder=encoder, decoder=dec); set_special_ids(m.config, BOS); set_special_ids(m.generation_config, BOS); set_special_ids(m.decoder.config, BOS)
        m.config.vocab_size = VOCAB; s = Seq2SeqSum(m, "encdec"); s.new_param_names = {n for n, _ in s.named_parameters() if "crossattention" in n or "ln_cross_attn" in n}
        if verbose: print(f"encoder-decoder: NEW weights (cross-attention) = {sum(p.numel() for n, p in s.named_parameters() if n in s.new_param_names) / 1e6:.1f}M parameters, randomly initialised; everything else is Phase 1")
        del enc_full, dec_lm
    elif arch == "bilstm":
        lm, mc = load_full("bilstm"); s = BiLSTMSum(lm)
    else: raise ValueError(arch)
    if not hasattr(s, "new_param_names"): s.new_param_names = {n for n, _ in s.named_parameters() if n.startswith(("att.", "comb.", "dec."))} if arch == "bilstm" else set()
    s.arch = arch; s.n_layers = n_layers or 0; gc.collect(); return s

def budgets(arch, n_virtual=0):
    """(input tokens, summary tokens) per architecture; GPT-2 shares its 512 positions between the two"""
    if arch == "gpt2":
        out = int(max(32, min(GPT2_SUM_TOK, SUM_P75 if GPT2_SUM_FROM_DATA else GPT2_SUM_TOK))); return max(32, 512 - out - len(SEP_IDS) - 2 - n_virtual), out
    if arch == "t5": return T5_ENC_TOK, int(min(T5_SUM_TOK, max(64, SUM_P90)))
    if arch == "encdec": return ENCDEC_ENC_TOK, int(min(510, T5_SUM_TOK, max(64, SUM_P90)))
    if arch == "bilstm": return BILSTM_ENC_TOK, int(min(BILSTM_SUM_TOK, max(64, SUM_P75)))
    raise ValueError(arch)
def count_params(m, trainable=False): return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))


In [ ]:
# ================= CELL 7 — ADAPTATION TECHNIQUES  [pure code]: full, partial, LoRA, LoRMA+ / LoRMA-pi (Phase 5 code, unchanged), Houlsby / Pfeiffer bottleneck adapters, prompt tuning =================
# All are our own small implementations (CPU-testable, no extra library). Every parameter-efficient method is the IDENTITY at initialisation except prompt tuning (the virtual tokens change the input by design).
# NOT implemented: prefix tuning, AdaLoRA and QLoRA (QLoRA needs bitsandbytes on a GPU; neither could be tested). The rule check does not need them: see the experiment pool in the PART SETTINGS cell.
from transformers.pytorch_utils import Conv1D
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A.to(x.dtype)) @ self.B.to(x.dtype) * self.scale
class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B = 0 at init  (= W0 x + (alpha/r) B A (W0 x))
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t().to(h.dtype)) @ self.B.t().to(h.dtype)) * self.scale
class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0; A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A); i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)
class _Wrap:
    """expose the wrapped layer's weight / bias: some Hugging Face modules read them directly (T5's feed-forward dtype check does)"""
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self): return self.base.bias
def proj_dims(m):
    return (m.in_features, m.out_features) if isinstance(m, nn.Linear) else tuple(m.weight.shape)      # HF Conv1D weight is [in, out]
class FusedQV(_Wrap, nn.Module):                  # wraps a FUSED q|k|v projection (GPT-2 Conv1D c_attn, ModernBERT Linear Wqkv); LoRA / LoRMA adapt q and v
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); nx, nf = proj_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora": self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi": self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora": q, v = q + self.q(x), v + self.v(x)
        else: q, v = self.q(q), self.v(v)
        return torch.cat([q, k, v], -1)
class LinAdapt(_Wrap, nn.Module):                 # the same adapters for separate nn.Linear projections (T5's q and v, self- and cross-attention)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        else: raise ValueError(kind)
    def forward(self, x):
        y = self.base(x); return y + self.op(x) if self.kind == "lora" else self.op(y)
class BottleneckAdapter(nn.Module):               # Houlsby et al. (2019): h + W_up GELU(W_down h); W_up = 0 at init, so the adapter starts as the identity
    def __init__(self, d, r):
        super().__init__(); self.down, self.up = nn.Linear(d, r), nn.Linear(r, d); nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias); nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, h): return h + self.up(F.gelu(self.down(h)))
class AdapterAfter(_Wrap, nn.Module):             # inserts a bottleneck adapter right after a sub-layer's output projection (before the residual addition)
    def __init__(self, base, r):
        super().__init__(); self.base = base; self.adapter = BottleneckAdapter(proj_dims(base)[1], r)
    def forward(self, x): return self.adapter(self.base(x))
def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw) if kind != "adapter" else AdapterAfter(child, kw["r"])); n += 1
    assert n > 0, f"no nn.Linear named {names} found"
    return n

TECHNIQUES = ("full", "partial", "lora", "lorma_plus", "lorma_pi", "houlsby", "pfeiffer", "prompt")
FAMILY = {"full": "full fine-tuning", "partial": "partial fine-tuning", "lora": "LoRA", "lorma_plus": "LoRMA", "lorma_pi": "LoRMA", "houlsby": "Houlsby adapters", "pfeiffer": "Pfeiffer adapters", "prompt": "prompt tuning", "kd": "knowledge distillation"}
PEFT = ("lora", "lorma_plus", "lorma_pi", "houlsby", "pfeiffer", "prompt")
DEFAULT_LR = {"full": {"gpt2": 5e-5, "t5": 2e-4, "encdec": 5e-5, "bilstm": 5e-4}, "partial": {"gpt2": 5e-5, "t5": 2e-4, "encdec": 5e-5, "bilstm": 5e-4},
              "lora": 5e-4, "lorma_plus": 5e-4, "lorma_pi": 2e-4, "houlsby": 5e-4, "pfeiffer": 5e-4, "prompt": 3e-3}           # starting points (T5 conventionally fine-tunes with a larger rate in fp32)
NEW_WEIGHT_LR = 3e-4                                                                     # the randomly initialised cross-attention / attention layers (encdec, bilstm) when the rest uses a small full-fine-tuning rate
def _gpt2_blocks(s): return list(s.lm.transformer.h) if s.arch == "gpt2" else list(s.model.decoder.transformer.h)
def apply_technique(s, technique, hp=None):
    """freeze / wrap the summariser in place. New weights of the architecture itself (encdec cross-attention, bilstm attention) are always trained."""
    hp = hp or {}; assert technique in TECHNIQUES, technique; a = s.arch
    if technique == "full":
        for p in s.parameters(): p.requires_grad = True
        return s
    for p in s.parameters(): p.requires_grad = False
    if technique == "partial":                                                           # embeddings + the lowest `frac` of the layers frozen, the rest trained
        frac = hp.get("frac", 0.5)
        for n, p in s.named_parameters(): p.requires_grad = True
        if a == "bilstm":
            k = int(s.lm.layers * frac)
            for p in s.lm.emb.parameters(): p.requires_grad = False
            for n, p in list(s.lm.fwd.named_parameters()) + list(s.lm.bwd.named_parameters()):
                if int(n.split("_l")[-1].split("_")[0]) < k: p.requires_grad = False
        else:
            if a == "gpt2": layers, low = list(s.lm.transformer.h), [s.lm.transformer.wte, s.lm.transformer.wpe]
            elif a == "t5": layers, low = list(s.model.encoder.block) + list(s.model.decoder.block), [s.model.shared]
            else: layers, low = list(s.model.encoder.layers), [s.model.encoder.embeddings]
            for mod in low + layers[: int(len(layers) * frac)]:
                for p in mod.parameters(): p.requires_grad = False
    elif technique == "prompt":
        assert a == "gpt2", "prompt tuning is implemented for the decoder-only GPT-2 summariser only"; s.enable_prompt(hp.get("n_virtual", 20))
    elif a == "bilstm": raise NotImplementedError(f"'{technique}' needs attention / linear projection layers: the BiLSTM supports 'full' and 'partial' only")
    else:
        kind = "adapter" if technique in ("houlsby", "pfeiffer") else technique; r = hp.get("r", 16 if kind != "adapter" else 64)
        kw = dict(r=r, alpha=hp.get("alpha", 2 * r), drop=hp.get("adapter_dropout", 0.05))
        if a in ("gpt2", "encdec"):
            for blk in _gpt2_blocks(s):
                if kind == "adapter":
                    if technique == "houlsby": blk.attn.c_proj = AdapterAfter(blk.attn.c_proj, r)
                    blk.mlp.c_proj = AdapterAfter(blk.mlp.c_proj, r)
                else: assert isinstance(blk.attn.c_attn, Conv1D), "unexpected c_attn type in this transformers version"; blk.attn.c_attn = FusedQV(blk.attn.c_attn, kind, **kw)
        if a == "encdec":
            for lay in s.model.encoder.layers:
                if kind == "adapter":
                    if technique == "houlsby": lay.attn.Wo = AdapterAfter(lay.attn.Wo, r)
                    lay.mlp.Wo = AdapterAfter(lay.mlp.Wo, r)
                else: assert isinstance(lay.attn.Wqkv, nn.Linear); lay.attn.Wqkv = FusedQV(lay.attn.Wqkv, kind, **kw)
        if a == "t5":
            if kind == "adapter": wrap_linear_targets(s.model, "adapter", ("o", "wo") if technique == "houlsby" else ("wo",), r=r)
            else: wrap_linear_targets(s.model, kind, ("q", "v"), **kw)
    for n, p in s.named_parameters():                                                     # the architecture's own new weights always train
        if n in s.new_param_names or "crossattention" in n or "ln_cross_attn" in n or n.startswith(("att.", "comb.")): p.requires_grad = True
    return s


In [ ]:
# ================= CELL 8 — TRAINER, GENERATION AND EVALUATION  [pure code; needs torch]: any technique, optional knowledge distillation, mixed precision, gradient accumulation, EMA, time caps, checkpoints, resumable runs =================
# Loss = token cross-entropy on the SUMMARY tokens only. Distillation (kd=...) = alpha * CE + (1 - alpha) * T^2 * KL(teacher || student) at every summary position (token level), trained on the reference
# summaries PLUS teacher-generated summaries of part of the training set (sequence level). Validation numbers come from generated text scored against the held-out references; test references are never used.
def use_amp_for(arch): return AMP and not (arch == "t5" and not T5_FP16)
def amp_ctx(on): return torch.autocast("cuda", dtype=torch.float16) if (on and device.type == "cuda") else contextlib.nullcontext()
class EMA:
    """exponential moving average of the trainable weights (compared with the raw weights on validation loss at the end; the better one is kept)"""
    def __init__(self, model, decay):
        self.params = [p for p in model.parameters() if p.requires_grad]; self.shadow = [p.detach().clone() for p in self.params]; self.decay = decay; self.backup = None
    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step))
        for s_, p in zip(self.shadow, self.params): s_.mul_(d).add_(p.detach(), alpha=1 - d)
    @torch.no_grad()
    def apply(self):
        self.backup = [p.detach().clone() for p in self.params]
        for p, s_ in zip(self.params, self.shadow): p.copy_(s_)
    @torch.no_grad()
    def restore(self):
        for p, b in zip(self.params, self.backup): p.copy_(b)
        self.backup = None
def ckpt_files(name): return CKPT / f"{name}.pt", CKPT / f"{name}.json"
def ckpt_exists(name): return ckpt_files(name)[0].exists() and ckpt_files(name)[1].exists()
def to_dev(b): return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}

INPUT_CACHE = {}
def inputs_for(D, name, idxs, strategy, budget):
    key = (name, strategy, budget)
    if key not in INPUT_CACHE: INPUT_CACHE[key] = {}
    c = INPUT_CACHE[key]
    for i in idxs:
        if i not in c: c[i] = build_input(D, i, strategy, budget)
    return [c[i] for i in idxs]
def seq_loss(s, b, teacher=None, kd=None, ls=0.0, amp_on=False):
    with amp_ctx(amp_on): lg, lab = s.forward_logits(b)
    m = lab != -100; z = lg[m].float(); ce = F.cross_entropy(z, lab[m], label_smoothing=ls)
    if teacher is None: return ce, ce
    with torch.no_grad(), amp_ctx(amp_on and teacher.arch != "t5"): tz = teacher.forward_logits(b)[0][m].float()
    T_ = kd.get("T", 2.0); kl = F.kl_div(F.log_softmax(z / T_, -1), F.log_softmax(tz / T_, -1), log_target=True, reduction="batchmean") * T_ * T_
    a_ = kd.get("alpha", 0.5); return a_ * ce + (1 - a_) * kl, ce
@torch.no_grad()
def val_loss(s, srcs, tgts, mb, amp_on):
    s.eval(); tot, n = 0.0, 0
    for k in range(0, len(srcs), mb):
        b = to_dev(s.make_batch(srcs[k:k + mb], tgts[k:k + mb])); _, ce = seq_loss(s, b, amp_on=amp_on); w = int((b["labels"] != -100).sum()); tot += float(ce) * w; n += w
    return tot / max(1, n)

def postprocess(text, dedupe=True):
    t = re.sub(r"\s+", " ", text or "").strip()
    if not dedupe or not t: return t
    out, seen = [], []
    for sent in re.split(r"(?<=[.?!;])\s+", t):                                         # redundancy reduction: drop a sentence whose words mostly repeat an earlier one
        w = set(rouge_tokens(sent))
        if not w or any(len(w & v) / len(w) > 0.8 for v in seen): continue
        seen.append(w); out.append(sent)
    return " ".join(out)
GEN_BS = {"gpt2": 16, "t5": 8, "encdec": 16, "bilstm": 16}
@torch.no_grad()
def _gen_batched(s, srcs, g):
    s.eval(); order = sorted(range(len(srcs)), key=lambda i: -len(srcs[i])); res = [None] * len(srcs); bs = max(1, GEN_BS[s.arch] // max(1, int(g.get("beams", 1))))
    for k in range(0, len(order), bs):
        sel = order[k:k + bs]
        with amp_ctx(use_amp_for(s.arch)): ids = s.generate_ids([srcs[i] for i in sel], g)
        for i, o in zip(sel, ids): res[i] = o
    return res
def gen_texts(s, D, name, idxs, strategy, g, in_budget):
    """generate summaries for documents idxs of split D. strategy = an input strategy, or 'hier2:<s>' (chunk summaries -> second-stage summary) or 'concat:<s>' (chunk summaries joined)"""
    t0 = time.time(); mode, _, base = strategy.partition(":") if ":" in strategy else ("single", "", strategy)
    if mode == "single": outs = [detok(o) for o in _gen_batched(s, inputs_for(D, name, idxs, strategy, in_budget), g)]
    else:
        chunks = [chunk_windows(D, i, in_budget, overlap=g.get("overlap", 0.25), max_chunks=g.get("max_chunks", 6)) for i in idxs]; flat = [c for cs in chunks for c in cs]
        gc_ = {**g, "max_new": int(g.get("chunk_new", max(32, int(g["max_new"]) // 2))), "min_new": 1}; cs_out = [detok(o) for o in _gen_batched(s, flat, gc_)]; outs, p = [], 0
        joined = []
        for cs in chunks: joined.append(" ".join(cs_out[p:p + len(cs)])); p += len(cs)
        if mode == "concat": outs = joined
        else: outs = [detok(o) for o in _gen_batched(s, [np.asarray(enc(j)[:in_budget], dtype=np.int64) for j in joined], g)]
    outs = [postprocess(o, g.get("dedupe", True)) for o in outs]
    empty = [k for k, o in enumerate(outs) if not o]
    if empty and mode == "single":                                                        # an empty output is regenerated with the end token blocked for 16 steps (still generated text)
        again = _gen_batched(s, inputs_for(D, name, [idxs[k] for k in empty], strategy, in_budget), {**g, "min_new": 16})
        for k, o in zip(empty, again): outs[k] = postprocess(detok(o), g.get("dedupe", True))
    n_empty = sum(1 for o in outs if not o); outs = [o if o else "." for o in outs]
    return outs, (time.time() - t0) / max(1, len(idxs)), n_empty

def by_length(df, lens):
    long_ = np.asarray(lens) > LONG_THRESHOLD; out = {}
    for tag, m in (("short", ~long_), ("long", long_)):
        for k in ("bertscore_f1", "rouge1_f", "rougeL_f", "fact_coverage"): out[f"{k}_{tag}"] = float(np.nanmean(df[k].values[m])) if m.any() and df[k].notna().any() else float("nan")
    return out
def peak_gb(): return round(torch.cuda.max_memory_allocated() / 2**30, 2) if device.type == "cuda" else float("nan")
def reset_peak():
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
def save_run(name, res, s, meta, val_ids, preds, df):
    rd = WORK / "runs" / name; rd.mkdir(parents=True, exist_ok=True)
    json.dump([{"id": to_py(i), "summary": p} for i, p in zip(val_ids, preds)], open(rd / "val_preds.json", "w", encoding="utf-8"), ensure_ascii=False); df.assign(id=[str(i) for i in val_ids]).to_csv(rd / "val_scores.csv", index=False)
    if s is not None and SAVE_MODELS:
        peft = meta["technique"] in PEFT; state = {k: v.detach().cpu() for k, v in s.state_dict().items() if (not peft or k in {n for n, p in s.named_parameters() if p.requires_grad})}
        pth, mp = ckpt_files(name); torch.save({"state": state, "peft_only": peft}, pth); json.dump(meta, open(mp, "w"), indent=1); res["ckpt_mb"] = round(pth.stat().st_size / 2**20, 1)
    json.dump(res, open(rd / "result.json", "w"), indent=1)
    led = pd.concat([pd.read_csv(LEDGER).query("name != @name"), pd.DataFrame([res])]) if LEDGER.exists() else pd.DataFrame([res]); led.to_csv(LEDGER, index=False)
def load_finetuned(name):
    """rebuild a fine-tuned summariser: Phase 1 weights from MODEL_PATHS + the saved state (adapters only for parameter-efficient runs)"""
    pth, mp = ckpt_files(name); meta = json.load(open(mp)); ck = torch.load(pth, map_location="cpu", weights_only=False)
    s = make_summarizer(meta["arch"], meta.get("n_layers") or None, verbose=False); apply_technique(s, meta["base_technique"], meta.get("hp"))
    missing, unexpected = s.load_state_dict(ck["state"], strict=False); tr = {n for n, p in s.named_parameters() if p.requires_grad}
    assert not unexpected and (not ck.get("peft_only") or all(k not in tr for k in missing)), f"checkpoint {name} does not match: unexpected {unexpected[:3]}, missing trainable {[k for k in missing if k in tr][:3]}"
    return s.to(device).eval(), meta

def run_experiment(name, arch, technique="full", hp=None, strategy=None, epochs=3.0, max_minutes=60, lr=None, batch_docs=16, wd=0.01, warmup=0.06, ema_decay=0.999, ls=0.0, seed=SEED,
                   n_layers=None, kd=None, gen=None, control=False, exp_id="", max_train_docs=None, val_cap=None, test_gen=None, write=True, evals=4):
    """one experiment = (Phase 1 architecture, technique). control=True marks an ablation / study run that is never counted. kd=dict(teacher=<run name>, alpha, T, seq_frac, seq_minutes) makes it a distillation run."""
    rd = WORK / "runs" / name
    if write and not RERUN and (rd / "result.json").exists() and (ckpt_exists(name) or not SAVE_MODELS):
        r = json.load(open(rd / "result.json")); print(f"[{name}] already finished (restored or earlier in this session): {sel_metric(r)[1]} {sel_metric(r)[0]:.4f}. Set RERUN = True to train it again."); return r
    t_start = time.time(); seed_all(seed); rng = np.random.default_rng(seed); hp = dict(hp or {}); strategy = strategy or DEFAULT_STRATEGY; gen = {**DEFAULT_GEN, **(gen or {})}
    teacher = None
    if kd:
        assert ckpt_exists(kd["teacher"]), f"teacher checkpoint {kd['teacher']} not found (restore its notebook output first)"; teacher, tmeta = load_finetuned(kd["teacher"])
        assert tmeta["arch"] == arch, f"the student must share the teacher's architecture ({tmeta['arch']}) so the token distributions align"; strategy = tmeta["strategy"]
        for p in teacher.parameters(): p.requires_grad = False
    s = make_summarizer(arch, n_layers); apply_technique(s, technique, hp); s = s.to(device); nv = s.n_virtual() if hasattr(s, "n_virtual") else 0
    in_b, out_b = budgets(arch, nv); amp_on = use_amp_for(arch); single = ":" not in strategy; train_strategy = strategy.split(":")[-1]
    n_par, n_tr = count_params(s), count_params(s, True)
    tr_idx = np.arange(len(TRAIN["ids"])) if not max_train_docs else np.sort(rng.choice(len(TRAIN["ids"]), min(max_train_docs, len(TRAIN["ids"])), replace=False))
    srcs = inputs_for(TRAIN, "train", list(tr_idx), train_strategy, in_b); tgts = [TRAIN["sum_toks"][i][:out_b] for i in tr_idx]
    if kd and kd.get("seq_frac", 0) > 0:                                                # SEQUENCE-LEVEL distillation: teacher-generated summaries become extra targets (cached for resumption)
        cp = WORK / "cache" / f"kdseq_{kd['teacher']}_{len(tr_idx)}_{out_b}.pkl"
        if cp.exists(): pseudo = pickle.load(open(cp, "rb")); print(f"   teacher summaries restored from cache: {len(pseudo)}")
        else:
            pick = rng.permutation(len(tr_idx))[: int(kd["seq_frac"] * len(tr_idx))]; pseudo, t0 = {}, time.time()
            for k in range(0, len(pick), 64):
                sel = pick[k:k + 64]; outs = _gen_batched(teacher, [srcs[j] for j in sel], {**gen, "max_new": out_b, "beams": 1, "sample": False})
                for j, o in zip(sel, outs): pseudo[int(j)] = np.asarray([x for x in o if x not in SPECIAL_IDS][:out_b], dtype=np.int32)
                if time.time() - t0 > 60 * kd.get("seq_minutes", 30): print(f"   teacher generation stopped at its time cap after {len(pseudo)} documents"); break
            pickle.dump(pseudo, open(cp, "wb")); print(f"   teacher summaries generated for {len(pseudo)} training documents in {(time.time() - t0) / 60:.1f} min")
        keys = [j for j in pseudo if len(pseudo[j]) > 0]; srcs = srcs + [srcs[j] for j in keys]; tgts = tgts + [pseudo[j] for j in keys]
    mb = max(1, TOK_BUDGET[arch] // (in_b + out_b)); accum = max(1, math.ceil(batch_docs / mb)); docs_step = mb * accum
    tr_params = [p for n, p in s.named_parameters() if p.requires_grad and n not in s.new_param_names]; new_params = [p for n, p in s.named_parameters() if p.requires_grad and n in s.new_param_names]
    lr = lr or (DEFAULT_LR[technique][arch] if isinstance(DEFAULT_LR[technique], dict) else DEFAULT_LR[technique])
    groups = ([{"params": tr_params, "lr": lr}] if tr_params else []) + ([{"params": new_params, "lr": max(lr, NEW_WEIGHT_LR)}] if new_params else [])
    opt = torch.optim.AdamW(groups, weight_decay=wd); base_lrs = [g["lr"] for g in groups]; scaler = torch.amp.GradScaler("cuda", enabled=amp_on); ema = EMA(s, ema_decay)
    if GRAD_CKPT and arch != "bilstm":
        try: (s.lm if arch == "gpt2" else s.model).gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except Exception as e: print("   gradient checkpointing not enabled:", type(e).__name__)
    vi = [EVAL_IDX[k] for k in range(min(64, len(EVAL_IDX)))]; v_src = inputs_for(VAL, "val", vi, train_strategy, in_b); v_tgt = [VAL["sum_toks"][i][:out_b] for i in vi]
    def batches(epoch_seed):
        r = np.random.default_rng(epoch_seed); perm = r.permutation(len(srcs)); pool = 64 * mb; mbs = []
        for k in range(0, len(perm), pool):
            chunk = sorted(perm[k:k + pool], key=lambda j: len(srcs[j]) + len(tgts[j])); mbs += [chunk[q:q + mb] for q in range(0, len(chunk), mb)]
        r.shuffle(mbs); return mbs
    s.train(); reset_peak(); times = []; it = iter(batches(seed))
    for j in range(3):                                                                    # throughput probe (forward + backward); the first is warm-up
        t0 = time.time(); sel = next(it, None)
        if sel is None: it = iter(batches(seed + 100 + j)); sel = next(it)
        b = to_dev(s.make_batch([srcs[q] for q in sel], [tgts[q] for q in sel])); loss, _ = seq_loss(s, b, teacher, kd, ls, amp_on); scaler.scale(loss).backward(); opt.zero_grad(set_to_none=True)
        if device.type == "cuda": torch.cuda.synchronize()
        times.append(time.time() - t0)
    sec_step = float(np.mean(times[1:])) * accum; steps = max(5, int(math.ceil(epochs * len(srcs) / docs_step))); cap_steps = max(5, int(max_minutes * 60 * 0.95 / max(sec_step, 1e-3))); steps = min(steps, cap_steps)
    print(f"[{name}] {arch} | technique {'kd (' + technique + ')' if kd else technique} | strategy {strategy} | input {in_b} + summary {out_b} tokens | {n_par / 1e6:.1f}M params, {n_tr / 1e6:.2f}M trained ({n_tr / n_par:.2%}) | lr {lr:g} | "
          f"{'fp16 autocast' if amp_on else 'fp32'} | {len(srcs)} training pairs\n   plan: {steps} steps x {docs_step} pairs = {steps * docs_step / len(srcs):.2f} epochs | {sec_step:.2f}s/step -> ~{steps * sec_step / 60:.0f} min training (cap {max_minutes} min)")
    hist, run_l, step, ep, t_train = [], 0.0, 0, 0, time.time(); it = iter(batches(seed + 1)); evals_at = {int(round(steps * (k + 1) / evals)) for k in range(evals)}
    while step < steps:
        step += 1; w = min(step / max(1, int(warmup * steps)), (steps - step + 1) / max(1, steps - int(warmup * steps)), 1.0)
        for g_, b0 in zip(opt.param_groups, base_lrs): g_["lr"] = b0 * w
        for _ in range(accum):
            try: sel = next(it)
            except StopIteration: ep += 1; it = iter(batches(seed + 1 + ep)); sel = next(it)
            b = to_dev(s.make_batch([srcs[q] for q in sel], [tgts[q] for q in sel])); loss, ce = seq_loss(s, b, teacher, kd, ls, amp_on); scaler.scale(loss / accum).backward(); run_l += float(ce) / accum
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for g_ in groups for p in g_["params"]], 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(step)
        if step % max(1, steps // 10) == 0: print(f"   step {step}/{steps} | train CE {run_l / max(1, steps // 10):.4f} | {(time.time() - t_start) / 60:.1f} min", flush=True); run_l = 0.0
        if step in evals_at: ema.apply(); vl = val_loss(s, v_src, v_tgt, mb, amp_on); ema.restore(); s.train(); hist.append((step, round(vl, 4))); print(f"   [validation loss, EMA weights, {len(vi)} docs] step {step}: {vl:.4f}", flush=True)
        if time.time() - t_train > max_minutes * 60: print(f"   time cap reached ({max_minutes} min) at step {step}/{steps}"); break
    train_min = (time.time() - t_train) / 60; peak_train = peak_gb()
    vl_raw = val_loss(s, v_src, v_tgt, mb, amp_on); ema.apply(); vl_ema = val_loss(s, v_src, v_tgt, mb, amp_on); weights = "ema"
    if vl_raw < vl_ema: ema.restore(); weights = "raw"
    del ema, opt; gc.collect()
    if teacher is not None: del teacher; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    print(f"   validation loss: raw {vl_raw:.4f} | EMA {vl_ema:.4f} -> keeping the {weights} weights")
    ev = EVAL_IDX[: (val_cap or len(EVAL_IDX))]; reset_peak(); g_out = {**gen, "max_new": gen.get("max_new") or out_b}
    preds, sec_doc, n_empty = gen_texts(s, VAL, "val", ev, strategy, g_out, in_b); peak_gen = peak_gb()
    refs = [VAL["sums"][i] for i in ev]; df, m = score_summaries(preds, refs, [VAL["docs"][i] for i in ev]); bl = by_length(df, [VAL["doc_len"][i] for i in ev]); sm, sk = sel_metric(m)
    res = dict(name=name, part=PART, arch=arch, model=ARCH_FAMILY[arch], technique="kd" if kd else technique, base_technique=technique, kd_teacher=(kd or {}).get("teacher", ""), control=bool(control), exp_id=exp_id,
               strategy=strategy, in_tokens=in_b, sum_tokens=out_b, n_layers=n_layers or 0, params_M=round(n_par / 1e6, 2), trainable_M=round(n_tr / 1e6, 3), lr=lr, steps=step, epochs=round(step * docs_step / len(srcs), 2),
               train_minutes=round(train_min, 1), weights=weights, val_loss=round(min(vl_raw, vl_ema), 4), history=str(hist), gen=json.dumps(g_out), gen_sec_per_doc=round(sec_doc, 3), empty_outputs=n_empty,
               peak_mem_train_gb=peak_train, peak_mem_gen_gb=peak_gen, eval_sig=EVAL_SIG, val_docs=len(ev), sel_metric=sk, sel_value=sm, **{k: m[k] for k in ("bertscore_f1", "rouge1_f", "rougeL_f", "fact_coverage", "decision_agree", "unsupported", "repetition3", "len_ratio", "pred_words")},
               rouge_impl=m["rouge_impl"], bertscore_setting=m["bertscore_setting"], **bl, test_written=False, minutes=0.0, ckpt_mb=0.0)
    print(f"[{name}] VALIDATION ({len(ev)} docs; NOT official test scores): BERTScore F1 {m['bertscore_f1']:.4f} | ROUGE-1 F1 {m['rouge1_f']:.4f} | ROUGE-L F1 {m['rougeL_f']:.4f} | fact coverage {m['fact_coverage']:.3f} | "
          f"length ratio {m['len_ratio']:.2f} | repetition {m['repetition3']:.3f} | unsupported {m['unsupported']:.3f} | {sec_doc:.2f} s/doc | empty {n_empty}")
    test_gen = TEST_GEN if test_gen is None else test_gen
    if write and test_gen:
        est = sec_doc * len(TEST["ids"]) / 60
        if est > TEST_GEN_MAX_MIN: print(f"   test generation SKIPPED: estimated {est:.0f} min > TEST_GEN_MAX_MIN = {TEST_GEN_MAX_MIN} (the final part generates the chosen model's test file)")
        else:
            tp, tsec, te_empty = gen_texts(s, TEST, "test", list(range(len(TEST["ids"]))), strategy, g_out, in_b); path = SUBDIR / f"{name}.json"
            write_submission(path, TEST["ids"], tp); validate_submission(path, TEST["ids"], TEST["docs"]); res["test_written"] = True
            print(f"   test submission {path} ({len(tp)} summaries, format validated, {tsec * len(tp) / 60:.1f} min, empty outputs {te_empty})")
    res["minutes"] = round((time.time() - t_start) / 60, 1)
    if write:
        fp_idx = EVAL_IDX[:4]; fingerprint = val_loss(s, inputs_for(VAL, "val", fp_idx, train_strategy, in_b), [VAL["sum_toks"][i][:out_b] for i in fp_idx], 4, amp_on)       # reload check: the same loss must come back
        meta = dict(arch=arch, technique=res["technique"], base_technique=technique, hp=hp, n_layers=n_layers or 0, strategy=strategy, in_tokens=in_b, sum_tokens=out_b, gen=g_out, weights=weights, fingerprint=fingerprint)
        save_run(name, res, s, meta, [VAL["ids"][i] for i in ev], preds, df); print(f"   saved runs/{name}/ (result, validation summaries, per-document scores){', checkpoint ' + str(res['ckpt_mb']) + ' MiB' if SAVE_MODELS else ''} and the ledger row | {res['minutes']} min in total")
    del s; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return res

def run_or_skip(name, arch, **kw):
    if not arch_available(arch): refresh_models(verbose=False)
    if not arch_available(arch):
        print(f"SKIPPED {name}: architecture '{arch}' needs the Phase 1 checkpoint(s) {[k for k in ARCH_NEEDS[arch] if k not in MODEL_PATHS]}, which are not attached (found: {sorted(MODEL_PATHS)}). Nothing else depends on it."); return None
    return run_experiment(name, arch, **kw)
def safe(label, fn):
    try: return fn()
    except Exception:
        print(f"{label} FAILED, continuing with the next cell:"); traceback.print_exc(); gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return None
def ledger():
    return pd.read_csv(LEDGER) if LEDGER.exists() else pd.DataFrame(columns=["name"])


In [ ]:
# ================= RESTORE EARLIER PARTS  [NEEDS the earlier parts' notebook outputs attached (Add Input > Notebook Output Files); several may be attached, they are merged]. =================
# Merges EVERY source it finds: run folders, checkpoints, submissions, the ledger (rows merged by name, later sources win), the long-document decision, the generation study and the distillation caches.
# A run that is restored is NOT trained again (RERUN = False), which is what makes the chain resumable: re-running a crashed part only redoes the runs that did not finish.
RESTORE_ROOTS = globals().get("RESTORE_ROOTS", ["/kaggle/input"])
srcs = sorted({str(Path(p).parent) for r in RESTORE_ROOTS for p in glob.glob(f"{r}/**/p6/ledger.csv", recursive=True)} - {str(WORK)})
srcs += sorted({str(Path(p).parent) for r in RESTORE_ROOTS for p in glob.glob(f"{r}/**/p6/split.json", recursive=True)} - {str(WORK)} - set(srcs))
TOP_FILES = ("split.json", "best_long_doc_strategy.json", "longdoc_study.csv", "longdoc_input_coverage.csv", "baselines.csv", "compat_report.csv", "gen_study.csv", "gen_choice.json", "kd_report.json")
ledgers, n_files = [], 0
for src in srcs:
    src = Path(src)
    for sub in ("runs", "submissions", "checkpoints"):
        if (src / sub).exists(): shutil.copytree(src / sub, WORK / sub, dirs_exist_ok=True); n_files += sum(1 for p in (src / sub).rglob("*") if p.is_file())
    for fn in TOP_FILES:
        if (src / fn).exists() and not (WORK / fn).exists(): shutil.copy2(src / fn, WORK / fn); n_files += 1
    if (src / "split.json").exists() and json.load(open(src / "split.json")).get("val_ids_sha1") != split_info["val_ids_sha1"]:
        print(f"WARNING: {src} was made with a DIFFERENT validation split (other data or settings): its validation scores are not comparable with this session's")
    for f in glob.glob(str(src / "cache" / "kdseq_*.pkl")): shutil.copy2(f, WORK / "cache" / Path(f).name)
    if (src / "ledger.csv").exists(): ledgers.append(pd.read_csv(src / "ledger.csv"))
    print(f"restored from {src}")
if ledgers: pd.concat(([pd.read_csv(LEDGER)] if LEDGER.exists() else []) + ledgers).drop_duplicates("name", keep="last").to_csv(LEDGER, index=False)
if not srcs: print("nothing to restore: no earlier part's output is attached" + (" -- this part NEEDS one (see its introduction)" if globals().get("REQUIRE_RESTORE") else " (fine for this part)"))
else:
    led_ = ledger(); print(f"{n_files} files | ledger rows {len(led_)}: {sorted(led_.name)} | checkpoints: {sorted(p.stem for p in CKPT.glob('*.pt'))}")
    if "eval_sig" in led_:
        other = sorted(set(led_.eval_sig.dropna()) - {EVAL_SIG})
        if other: print(f"WARNING: some restored runs were scored on a different validation subset ({other} vs this session's {EVAL_SIG}): they are compared only after re-scoring (final part)")
if globals().get("REQUIRE_RESTORE"): assert srcs, "attach the earlier parts' notebook outputs (Add Input > Notebook Output Files) and run again"


In [ ]:
# ===== SHARED SETTINGS  [pure code; run it before the experiment cells]: lengths, precision, batch sizes, the long-document strategy and the default decoding (every value here is a hyperparameter, never a separate experiment) =====
GPT2_SUM_TOK, GPT2_SUM_FROM_DATA = 256, True          # GPT-2 summary budget: min(256, p75 of the training summaries); the document gets the rest of the 512 positions
T5_ENC_TOK, T5_SUM_TOK = 1024, 512                    # T5 reads up to 1,024 document tokens (relative positions); summary budget min(512, p90)
ENCDEC_ENC_TOK = 512                                  # ModernBERT encoder input (its Phase 1 context; rotary positions could read more, untested)
BILSTM_ENC_TOK, BILSTM_SUM_TOK = 512, 256
T5_FP16 = False                                       # T5 in fp32 (fp16 was unstable in Phases 4-5)
GRAD_CKPT = False                                     # gradient checkpointing (set True if a run runs out of GPU memory; slower)
TOK_BUDGET = {"gpt2": 8192, "t5": 4096, "encdec": 6144, "bilstm": 8192}           # tokens per micro-batch; gradient accumulation reaches batch_docs documents per optimiser step
if SELFTEST: TOK_BUDGET = {k: 1200 for k in TOK_BUDGET}; GPT2_SUM_TOK = 64; T5_ENC_TOK, T5_SUM_TOK = 256, 64; BILSTM_SUM_TOK = 64
DEFAULT_GEN = dict(max_new=None, min_new=8, beams=1, no_repeat=3, rep_pen=1.0, len_pen=1.0, sample=False, dedupe=True)   # greedy + trigram blocking + duplicate-sentence removal; max_new None = the architecture's summary budget
DEFAULT_STRATEGY_FALLBACK = "head_tail"               # documented default when Part 1's long-document study is missing: the start (facts) and the end (decision) of the judgment
_ld = WORK / "best_long_doc_strategy.json"
DEFAULT_STRATEGY = json.load(open(_ld))["strategy"] if _ld.exists() else DEFAULT_STRATEGY_FALLBACK
print(f"long-document input strategy for the experiments: '{DEFAULT_STRATEGY}' ({'chosen by Part 1 study: ' + str(_ld) if _ld.exists() else 'DEFAULT (Part 1 study result not attached)'})")
for a in ARCH_NEEDS:
    if arch_available(a): print(f"  {a:7s} budgets: input {budgets(a)[0]} tokens, summary {budgets(a)[1]} tokens")


In [ ]:
# ================= PART 1 — ARCHITECTURE COMPATIBILITY WITH THE GENERATION TASK  [NEEDS the Phase 1 checkpoints; a few minutes, CPU or GPU] =================
# For every attached checkpoint: what it is, its size and context, whether it can generate, and a sanity measurement that the weights loaded correctly (a language-model loss far below log(vocabulary) means they did).
# The BiLSTM output head is NOT defined in arch_loaders.py: the head assumed by BiLSTMSum (emb^T proj_f(norm_f(h)) + out_bias) is checked here; a loss near log(vocabulary) would mean the assumption is wrong.
@torch.no_grad()
def lm_loss_gpt2(lm, texts):
    lm.eval().to(device); tot, n = 0.0, 0
    for t in texts:
        ids = torch.tensor([[BOS] + enc(t)[:511]], device=device); out = lm(input_ids=ids, labels=ids); w = ids.shape[1] - 1; tot += float(out.loss) * w; n += w
    return tot / max(1, n)
@torch.no_grad()
def lm_loss_bilstm(lm, texts):
    s = BiLSTMSum(lm).to(device).eval(); tot, n = 0.0, 0
    for t in texts:
        ids = torch.tensor([[BOS] + enc(t)[:511]], device=device); d, _ = s.lm.fwd(s.lm.emb(ids[:, :-1])); lg = s.head(d).float(); l = F.cross_entropy(lg[0], ids[0, 1:]); w = ids.shape[1] - 1; tot += float(l) * w; n += w
    return tot / max(1, n)
@torch.no_grad()
def mlm_acc_modernbert(full, texts):
    full.eval().to(device); r = np.random.default_rng(0); hit, n = 0, 0
    for t in texts:
        ids = torch.tensor([enc(t)[:510]], device=device); m = torch.from_numpy(r.random(ids.shape[1]) < 0.15).to(device); m[0] = True; x = ids.clone(); x[0, m] = MASK
        lg = full(input_ids=x, attention_mask=torch.ones_like(x)).logits; hit += int((lg[0, m].argmax(-1) == ids[0, m]).sum()); n += int(m.sum())
    return hit / max(1, n)
COMPAT_TEXTS = [VAL["docs"][i][:4000] for i in EVAL_IDX[:8]]; rows = []
def compat(key):
    full, mc = load_full(key); fam = mc["family"]; row = dict(checkpoint=key, family=fam, class_=type(full).__name__, params_M=round(count_params(full) / 1e6, 1), context=mc.get("context_length"), hidden=mc.get("hidden_size"), step=None)
    if fam == "gpt2": row.update(generates="yes (decoder-only)", check="LM loss on validation documents", value=round(lm_loss_gpt2(full, COMPAT_TEXTS), 3), expected="about 3.2 (Phase 1 validation loss 3.24)")
    elif fam == "t5":
        row.update(generates="yes (native encoder-decoder)", decoder_start=getattr(full.config, "decoder_start_token_id", None) or getattr(full.generation_config, "decoder_start_token_id", None), t5_pad=full.config.pad_token_id, t5_eos=full.config.eos_token_id)
        s = make_summarizer("t5", verbose=False).to(device).eval(); b = to_dev(s.make_batch([build_input(VAL, EVAL_IDX[0], "head", 256)], [VAL["sum_toks"][EVAL_IDX[0]][:64]]))
        with torch.no_grad(): lg, lab = s.forward_logits(b)
        row.update(check="untrained summary CE (just runs) + 1 generation", value=round(float(F.cross_entropy(lg[lab != -100].float(), lab[lab != -100])), 3), expected="any finite value; the task is not learned yet")
        out = s.generate_ids([build_input(VAL, EVAL_IDX[0], "head", 256)], {**DEFAULT_GEN, "max_new": 20}); print("T5 zero-shot output (untrained task):", repr(detok(out[0])[:150])); del s
    elif fam == "modernbert": row.update(generates="NO by itself -> paired with the Phase 1 GPT-2 decoder (encdec)", check="masked-token accuracy (15% masked)", value=round(mlm_acc_modernbert(full, COMPAT_TEXTS), 3), expected="well above 0 (Phase 1 MLM loss 1.74)")
    elif fam == "bilstm": row.update(generates="yes, as seq2seq with new attention (weak)", check="forward-LM loss with the assumed head", value=round(lm_loss_bilstm(full, COMPAT_TEXTS), 3), expected=f"about 6 (Phase 1 5.96); log(V) = {math.log(VOCAB):.2f} would mean a wrong head")
    rows.append(row); del full; gc.collect()
for key in ("gpt2_new", "t5", "modernbert", "bilstm"):
    if key in MODEL_PATHS: safe(f"compatibility {key}", lambda key=key: compat(key))
    else: rows.append(dict(checkpoint=key, family=key, generates="NOT ATTACHED: skipped", check="-", value=None, expected="-"))
if arch_available("encdec"):
    def _ed():
        s = make_summarizer("encdec").to(device).eval(); b = to_dev(s.make_batch([build_input(VAL, EVAL_IDX[0], "head", 256)], [VAL["sum_toks"][EVAL_IDX[0]][:64]]))
        with torch.no_grad(): lg, lab = s.forward_logits(b)
        rows.append(dict(checkpoint="modernbert+gpt2_new", family="encdec", generates="yes (EncoderDecoderModel)", params_M=round(count_params(s) / 1e6, 1), check="forward runs; new cross-attention parameters (M)",
                         value=round(sum(p.numel() for n, p in s.named_parameters() if n in s.new_param_names) / 1e6, 2), expected="the only new weights")); del s
    safe("compatibility encdec", _ed)
COMPAT = pd.DataFrame(rows); print(COMPAT.to_string(index=False)); COMPAT.to_csv(WORK / "compat_report.csv", index=False)
gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


In [ ]:
# ================= PART 1 — BASELINES + INPUT-COVERAGE STUDY  [RUNS NOW: CPU only, no checkpoints]. Reference points only: none of these is a Phase 1 model, none is ever a submission. =================
# B1 lead-k: the first sentences of the document up to the median training-summary length.  B2 importance-k: the top-scoring sentences (the selector used by the 'importance' strategy), in document order.
# B3 ORACLE-k: the sentences that best match the reference: an UPPER BOUND for extractive selection (it reads the reference, so it can never be used on test documents).
# The input-coverage study answers 'is the information still in what the model reads?': for each long-document strategy and input length, the share of the reference's key facts that are present in
# the model's INPUT, and the ROUGE-1 recall of the reference by the input. It needs no model, so it isolates what truncation / chunking / selection lose before any training.
MED_SUM_WORDS = int(np.median([wc(s) for s in TRAIN["sums"]]))
def topk_text(D, i, words, scores):
    order = np.argsort(-scores, kind="stable"); keep, n = [], 0
    for k in order:
        keep.append(k); n += len(D["sents"][i][k].split())
        if n >= words: break
    return " ".join(D["sents"][i][k] for k in sorted(keep))
def oracle_scores(D, i): ref = set(rouge_tokens(D["sums"][i])); return np.array([len(set(rouge_tokens(s)) & ref) / math.sqrt(len(rouge_tokens(s)) + 3) for s in D["sents"][i]])
refs_ev = [VAL["sums"][i] for i in EVAL_IDX]; docs_ev = [VAL["docs"][i] for i in EVAL_IDX]; rows_b = []
for nm, fn in (("B1_lead_k", lambda i: lead_text(VAL, i, MED_SUM_WORDS)), ("B2_importance_k", lambda i: topk_text(VAL, i, MED_SUM_WORDS, VAL["sent_scores"][i])),
               ("B3_ORACLE_k (upper bound, reads the reference)", lambda i: topk_text(VAL, i, MED_SUM_WORDS, oracle_scores(VAL, i)))):
    preds_b = [fn(i) for i in EVAL_IDX]; df_b, m_b = score_summaries(preds_b, refs_ev, docs_ev); bl_b = by_length(df_b, [VAL["doc_len"][i] for i in EVAL_IDX])
    rows_b.append(dict(baseline=nm, **{k: round(m_b[k], 4) for k in ("bertscore_f1", "rouge1_f", "rougeL_f", "fact_coverage", "len_ratio", "repetition3")}, **{k: round(v, 4) for k, v in bl_b.items() if k.startswith(("rouge1", "bertscore"))}))
BASELINES = pd.DataFrame(rows_b); BASELINES.to_csv(WORK / "baselines.csv", index=False)
print(f"EXTRACTIVE BASELINES on the validation subset ({len(EVAL_IDX)} docs, k = {MED_SUM_WORDS} words = median training summary; reference points, NOT submissions):"); print(BASELINES.to_string(index=False))

COV_STRATS = ("head", "tail", "head_tail", "importance", "section", "oracle")
GPT2_IN = 512 - min(GPT2_SUM_TOK, max(32, SUM_P75)) - len(SEP_IDS) - 2
COV_BUDGETS = sorted({128, GPT2_IN, 512, 1024, 2048})
def recall_r1(ref, text): a, b = set(rouge_tokens(ref)), set(rouge_tokens(text)); return len(a & b) / max(1, len(a))
cov_rows, lens_ev = [], np.array([VAL["doc_len"][i] for i in EVAL_IDX]); ref_facts = [key_facts(VAL["sums"][i]) for i in EVAL_IDX]
for B in COV_BUDGETS:
    for st in COV_STRATS + ("chunks_all",):
        fc, r1 = [], []
        for k, i in enumerate(EVAL_IDX):
            txt = " ".join(detok(c) for c in chunk_windows(VAL, i, B)) if st == "chunks_all" else detok(build_input(VAL, i, st, B))
            fc.append(fact_coverage(ref_facts[k], txt)); r1.append(recall_r1(VAL["sums"][i], txt))
        fc, r1 = np.array(fc, float), np.array(r1); lg = lens_ev > LONG_THRESHOLD
        cov_rows.append(dict(budget=B, strategy=st if st != "chunks_all" else "overlapping chunks (all, <=6)", facts_in_input=np.nanmean(fc), facts_in_input_long_docs=np.nanmean(fc[lg]) if lg.any() else np.nan, ref_word_recall=r1.mean(), ref_word_recall_long_docs=r1[lg].mean() if lg.any() else np.nan))
COVERAGE = pd.DataFrame(cov_rows).round(3); COVERAGE.to_csv(WORK / "longdoc_input_coverage.csv", index=False)
print(f"\nINPUT COVERAGE (share of the reference's key facts / words that are still present in the model's input; budgets in tokens; GPT-2's input budget is {GPT2_IN}; 'oracle' reads the reference = upper bound):")
print(COVERAGE.pivot(index="strategy", columns="budget", values="facts_in_input").to_string()); print("\n(reference word recall by the input)"); print(COVERAGE.pivot(index="strategy", columns="budget", values="ref_word_recall").to_string())


In [ ]:
# ================= PART 1 — ZERO-SHOT REFERENCE + NEURAL LONG-DOCUMENT STUDY  [NEEDS the GPT-2 Phase 1 checkpoint + a GPU; about LD_MINUTES per strategy + generation]. ABLATIONS (control=True): never counted as experiments. =================
# One backbone (GPT-2, the fastest and the most context-limited, so the strategies matter most), one light full fine-tune PER input strategy with the same budget, scored on the same validation subset.
#   head / tail / head_tail / importance / section : WHICH 512-position input the model reads (a different fine-tune each)
#   head@128                                        : truncation length (the same 'head' strategy with a much shorter input)
#   hier2:<best> / concat:<best>                    : HIERARCHICAL inference with the best single-input model: overlapping chunks (25% overlap, sliding window, <= 6 chunks) are summarised one by one, then
#                                                     either a second-stage summary of the joined chunk summaries (hier2) or the joined chunk summaries themselves (concat, longer output). Same weights: inference-time hierarchy.
# Decision rule (written to best_long_doc_strategy.json): the documented default 'head_tail' is replaced only by a strategy whose selection metric is higher by LD_MARGIN AND whose paired-bootstrap 95% interval over documents is above 0.
LD_STRATEGIES = ["head", "tail", "head_tail", "importance", "section", "head@128"]
LD_MINUTES, LD_EPOCHS, LD_TRAIN_DOCS, LD_VAL_CAP, LD_MARGIN = 20, 2.0, 4000, 200, 0.005
if SELFTEST: LD_MINUTES, LD_EPOCHS, LD_TRAIN_DOCS, LD_VAL_CAP = 0.3, 1.0, 60, 8
LD_HIER = True
ZS_DOCS = 100 if not SELFTEST else 6
if arch_available("gpt2"):
    def zero_shot():
        s = make_summarizer("gpt2").to(device).eval(); ib, ob = budgets("gpt2"); ev = EVAL_IDX[:ZS_DOCS]
        preds, sec, ne = gen_texts(s, VAL, "val", ev, "head", {**DEFAULT_GEN, "max_new": ob}, ib); df_z, m_z = score_summaries(preds, [VAL["sums"][i] for i in ev], [VAL["docs"][i] for i in ev])
        res = dict(name="Z0_gpt2_zero_shot", part=PART, arch="gpt2", model="gpt2", technique="zero-shot", base_technique="none", control=True, strategy="head", val_docs=len(ev), eval_sig=EVAL_SIG + f"@{ZS_DOCS}",
                   **{k: m_z[k] for k in ("bertscore_f1", "rouge1_f", "rougeL_f", "fact_coverage", "len_ratio", "repetition3", "unsupported")}, gen_sec_per_doc=round(sec, 3), rouge_impl=m_z["rouge_impl"], bertscore_setting=m_z["bertscore_setting"])
        save_run(res["name"], res, None, {}, [VAL["ids"][i] for i in ev], preds, df_z)
        print(f"ZERO-SHOT Phase 1 GPT-2 (no fine-tuning, first {len(ev)} scored documents; a reference point): BERTScore F1 {m_z['bertscore_f1']:.4f} | ROUGE-1 {m_z['rouge1_f']:.4f} | ROUGE-L {m_z['rougeL_f']:.4f} | example: {preds[0][:200]!r}")
        del s; gc.collect()
    safe("zero-shot baseline", zero_shot)
    LD = {}
    for st in LD_STRATEGIES:
        LD[st] = safe(f"long-document study {st}", lambda st=st: run_or_skip(f"LD_gpt2_{st.replace('@', '_at')}", "gpt2", technique="full", strategy=st, epochs=LD_EPOCHS, max_minutes=LD_MINUTES, max_train_docs=LD_TRAIN_DOCS,
                                                                             val_cap=LD_VAL_CAP, control=True, test_gen=False))
    done = {st: r for st, r in LD.items() if r}
    if done:
        best_single = max((st for st in done if "@" not in st), key=lambda st: done[st]["sel_value"], default=None)
        if LD_HIER and best_single:
            nm = f"LD_gpt2_{best_single}"
            if ckpt_exists(nm):
                def hier(mode):
                    s, meta = load_finetuned(nm); ib, ob = meta["in_tokens"], meta["sum_tokens"]; ev = EVAL_IDX[:LD_VAL_CAP]; st = f"{mode}:{best_single}"; reset_peak()
                    preds, sec, ne = gen_texts(s, VAL, "val", ev, st, {**DEFAULT_GEN, "max_new": ob, "chunk_new": max(48, ob // 2)}, ib); df_h, m_h = score_summaries(preds, [VAL["sums"][i] for i in ev], [VAL["docs"][i] for i in ev]); sm, sk = sel_metric(m_h)
                    r = {**{k: v for k, v in done[best_single].items()}, "name": f"LD_gpt2_{mode}_{best_single}", "strategy": st, "sel_value": sm, "sel_metric": sk, "gen_sec_per_doc": round(sec, 3), "peak_mem_gen_gb": peak_gb(), "empty_outputs": ne,
                         **{k: m_h[k] for k in ("bertscore_f1", "rouge1_f", "rougeL_f", "fact_coverage", "decision_agree", "unsupported", "repetition3", "len_ratio", "pred_words")}, **by_length(df_h, [VAL["doc_len"][i] for i in ev]), "test_written": False}
                    save_run(r["name"], r, None, {}, [VAL["ids"][i] for i in ev], preds, df_h); done[st] = r; print(f"[{r['name']}] {sk} {sm:.4f} | ROUGE-1 {m_h['rouge1_f']:.4f} | fact coverage {m_h['fact_coverage']:.3f} | length ratio {m_h['len_ratio']:.2f} | {sec:.2f} s/doc")
                    del s; gc.collect()
                for mode in ("hier2", "concat"): safe(f"hierarchical {mode}", lambda mode=mode: hier(mode))
        tab = pd.DataFrame([dict(strategy=st, **{k: r.get(k) for k in ("sel_metric", "sel_value", "bertscore_f1", "rouge1_f", "rougeL_f", "fact_coverage", "len_ratio", "repetition3", "rouge1_f_short", "rouge1_f_long", "fact_coverage_long", "gen_sec_per_doc", "train_minutes")}) for st, r in done.items()])
        print("\nLONG-DOCUMENT STUDY (GPT-2, validation subset; NOT official test scores):"); print(tab.round(4).to_string(index=False)); tab.to_csv(WORK / "longdoc_study.csv", index=False)
        def per_doc(name_): p = WORK / "runs" / name_ / "val_scores.csv"; d = pd.read_csv(p); return (d["bertscore_f1"] if d["bertscore_f1"].notna().any() else 0.5 * (d["rouge1_f"] + d["rougeL_f"])).values
        ref = DEFAULT_STRATEGY_FALLBACK; chosen, evidence = ref, []
        if ref in done:
            base = per_doc(done[ref]["name"])
            for st, r in sorted(done.items(), key=lambda kv: -kv[1]["sel_value"]):
                if st == ref or ":" in st or "@" in st: continue                         # the experiments use a single-input strategy (hierarchical modes are evaluated in the final generation study)
                d, lo, hi, w = paired_bootstrap(per_doc(r["name"]), base); evidence.append(dict(strategy=st, diff_vs_default=round(d, 4), ci_low=round(lo, 4), ci_high=round(hi, 4)))
                if chosen == ref and d >= LD_MARGIN and lo > 0: chosen = st
        print("paired bootstrap against the default 'head_tail':", evidence, "\n=> strategy for all experiments:", chosen)
        json.dump(dict(strategy=chosen, default=ref, rule=f"replace the default only if better by {LD_MARGIN} with the paired 95% interval above 0", evidence=evidence, table=tab.round(4).to_dict("records")), open(WORK / "best_long_doc_strategy.json", "w"), indent=1, default=str)
        DEFAULT_STRATEGY = chosen
        for st, r in done.items():                                                   # keep only the checkpoint of the best single-input model (the others are ablations; saves output space)
            if r and st != best_single and ckpt_exists(r["name"]): [p.unlink() for p in ckpt_files(r["name"])]
else: print("SKIPPED the zero-shot reference and the neural long-document study: the GPT-2 Phase 1 checkpoint is not attached. Later parts use the documented default strategy 'head_tail'.")


In [ ]:
# ================= BACKUP  [NEEDS Kaggle credentials for the upload, otherwise a dry run]. The RELIABLE backup is this notebook's own output (/kaggle/working/p6): the next part restores it via Add Input > Notebook Output Files. =================
# Uploads by itself when the credentials cell verified the Kaggle key (KAGGLE_OK); otherwise a dry run that lists what would be uploaded. It never asks you to type anything. A CLI exit code is never trusted: 'datasets status' must say ready.
DATASET_SLUG = f"cs781-phase6-sum-{PART.lower().replace('_', '-')}"; INCLUDE_MODELS = True; DRY_RUN = SELFTEST or not globals().get("KAGGLE_OK", False); BACKUP_OK = False
OWNER = os.environ.get("KAGGLE_USERNAME", "oipranesh"); stage = Path("/kaggle/working/p6_backup") if str(WORK).startswith("/kaggle/working") else WORK.parent / "p6_backup"; shutil.rmtree(stage, ignore_errors=True); stage.mkdir(parents=True); files = {}
for rd in sorted((WORK / "runs").glob("*")):                                   # flat names (the Kaggle CLI skips sub-folders): <run>__<file>
    for f in sorted(rd.iterdir()): files[f"{rd.name}__{f.name}"] = f
for f in sorted(CKPT.glob("*")):
    if f.suffix == ".pt" and not INCLUDE_MODELS: continue
    files[f"ckpt__{f.name}"] = f
for f in sorted(SUBDIR.glob("*.json")): files[f"submission__{f.name}"] = f
for f in [LEDGER] + [WORK / fn for fn in TOP_FILES] + [Path("/kaggle/working/prediction.json"), WORK / "requirements_report.md", WORK / "analysis.md"]:
    if f.exists(): files[f"top__{f.name}"] = f
if not files: print("nothing to back up yet")
else:
    for name, p in files.items():
        try: os.link(p, stage / name)
        except OSError: shutil.copy2(p, stage / name)
    def sha256(p, chunk=1 << 24):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(chunk), b""): h.update(b)
        return h.hexdigest()
    runs_in = sorted({n.split("__")[0] for n in files if not n.startswith(("submission__", "top__", "ckpt__"))}); manifest = {"runs": runs_in, "files": {n: {"bytes": (stage / n).stat().st_size, "sha256": sha256(stage / n)} for n in files}}
    (stage / "MANIFEST.json").write_text(json.dumps(manifest, indent=1)); (stage / "dataset-metadata.json").write_text(json.dumps({"title": f"CS781 Phase 6 summarisation {PART} backup", "id": f"{OWNER}/{DATASET_SLUG}", "licenses": [{"name": "unknown"}]}, indent=1))
    tot = sum(p.stat().st_size for p in stage.iterdir()); print(f"staged {len(files) + 2} files, {tot / 2**30:.2f} GiB; runs: {runs_in}")
    kb = shutil.which("kaggle"); cmd_c = ["kaggle", "datasets", "create", "-p", str(stage)]; cmd_v = ["kaggle", "datasets", "version", "-p", str(stage), "-m", f"runs: {', '.join(runs_in)}"[:200]]
    if DRY_RUN or not kb: print("DRY RUN: would run", " ".join(cmd_c), "(or, if the dataset exists)", " ".join(cmd_v[:5]), "...\nThe notebook output (/kaggle/working/p6) is the backup the next part restores.")
    else:
        chk = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=180)
        if chk.returncode != 0: print("credentials check FAILED (Internet on? legacy key?):", (chk.stdout + chk.stderr)[:300], "\nNothing was uploaded.")
        else:
            def clean(txt): return "\n".join(l for l in txt.splitlines() if "%|" not in l and l.strip())
            BAD = r"(?i)creation error|slugs? .*null|\bwas not created\b|\bfailed\b|already (exists|in use)|\berror\b"
            r = subprocess.run(cmd_c, capture_output=True, text=True); o = (r.stdout + r.stderr).strip(); print(clean(o)[-900:])
            if r.returncode != 0 or re.search(BAD, o):                                        # the CLI can print an error and still exit 0: do not trust the exit code
                print("-> the create step reported a problem; trying a new VERSION of the dataset (it may already exist)"); r = subprocess.run(cmd_v, capture_output=True, text=True); print(clean((r.stdout + r.stderr).strip())[-900:])
            time.sleep(5); st = subprocess.run(["kaggle", "datasets", "status", f"{OWNER}/{DATASET_SLUG}"], capture_output=True, text=True, timeout=180); st_out = (st.stdout + st.stderr).strip()
            BACKUP_OK = st.returncode == 0 and "ready" in st_out.lower(); print(f"dataset status: {st_out[:100]!r}")
            print(f"\nBACKUP OK (verified: {OWNER}/{DATASET_SLUG} exists and is ready)" if BACKUP_OK else f"\nBACKUP NOT VERIFIED: the staged folder {stage} is kept; the notebook output remains the reliable backup.")


In [ ]:
# ================= WHAT 'Save & Run All' KEEPS AS THIS NOTEBOOK'S OUTPUT, a format check of every submission file, and a size cleanup  [RUNS NOW] =================
DELETE_BULKY = True       # remove the tokenisation cache and the staged backup copy (after a verified upload); results, generations, checkpoints and submissions are kept
def size_gb(p): return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 2**30 if Path(p).exists() else 0.0
print(f"{WORK} before cleanup: {size_gb(WORK):.2f} GiB (cache {size_gb(WORK / 'cache'):.2f}, checkpoints {size_gb(CKPT):.2f}, runs {size_gb(WORK / 'runs'):.2f}, submissions {size_gb(SUBDIR):.3f})")
if DELETE_BULKY:
    for f in (WORK / "cache").glob("tok_*.pkl"): f.unlink()
if globals().get("BACKUP_OK"): shutil.rmtree(stage, ignore_errors=True); print("backup uploaded and verified: the staged copy was removed")
print(f"after cleanup: {size_gb(WORK):.2f} GiB (Kaggle keeps at most 20 GB of output)\n\nSUBMISSION FILES (Option A: a JSON list of {{\"id\": ..., \"{PRED_KEY}\": \"...\"}}):")
for f in sorted(SUBDIR.glob("*.json")) + [p for p in (Path("/kaggle/working/prediction.json"),) if p.exists()]:
    try: validate_submission(f, TEST["ids"], TEST["docs"]); sub_ = json.load(open(f, encoding="utf-8")); print(f"  {f.name:40s} {len(sub_)} summaries | mean {np.mean([len(s[PRED_KEY].split()) for s in sub_]):.0f} words | valid")
    except AssertionError as e: print(f"  {f.name:40s} INVALID: {e}")
led_ = ledger()
if len(led_): print("\nLEDGER (validation only; NOT official test scores):"); print(led_[[c for c in ("name", "part", "arch", "technique", "control", "strategy", "bertscore_f1", "rouge1_f", "rougeL_f", "fact_coverage", "minutes", "test_written") if c in led_]].round(4).to_string(index=False))
print("\ncheckpoints:", sorted(p.name for p in CKPT.glob("*.pt")))
